In [ ]:
import subprocess
import sys

def pip(*args):
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *args],
        check=True
    )

# Install FAISS separately first (PyPI package name changed)
try:
    pip("faiss-gpu-cu12")  # Kaggle T4 runs CUDA 12
except Exception:
    pip("faiss-cpu")  # fallback if GPU wheel unavailable

# Install main dependencies
pip(
    "transformers==4.49.0",
    "colpali-engine>=0.3.1",
    "accelerate>=0.26.0",
    "bitsandbytes>=0.43.0",
    "einops",
    "pdf2image",
    "pillow>=10.0",
    "langdetect",
    "qwen-vl-utils",
    "sentence-transformers==2.2.2",
    "rank_bm25",
    "PyPDF2",
    "pdfplumber",
    "langgraph",
    "langchain",
    "plotly",
    "kaleido",
    "--upgrade",
)

# Install PyTorch (CUDA 11.8 compatible)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "torch",
        "torchvision",
        "torchaudio",
        "--index-url",
        "https://download.pytorch.org/whl/cu118",
    ],
    check=True
)

# Install poppler (required for pdf2image)
subprocess.run(
    ["apt-get", "-qq", "install", "-y", "poppler-utils"],
    check=True
)
!pip install pymupdf


print("✅ All dependencies installed")

In [1]:
import os, gc, warnings, time
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from dataclasses import dataclass

import torch
import numpy as np
from PIL import Image

warnings.filterwarnings("ignore")

# ── Device ──────────────────────────────────────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on: {DEVICE.upper()}")
if DEVICE == "cuda":
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {props.name}  |  VRAM: {props.total_memory/1e9:.1f} GB")

# ── Model IDs ────────────────────────────────────────────────────────────────
GEN_MODEL_NAME    = "Qwen/Qwen2.5-7B-Instruct"
EMBED_MODEL_NAME  = "BAAI/bge-base-en-v1.5"
RERANK_MODEL_NAME = "cross-encoder/ms-marco-TinyBERT-L2-v2"
COLPALI_ID        = "vidore/colpali-v1.2"
QWEN_VL_ID        = "Qwen/Qwen2.5-VL-7B-Instruct"

# ── Budget constants ──────────────────────────────────────────────────────────
MAX_CONTEXT_TOKENS = 2048
ANSWER_BUDGET      = 220
CONTEXT_BUDGET     = MAX_CONTEXT_TOKENS - ANSWER_BUDGET
COLPALI_TOP_K      = 3
MAX_NEW_TOKENS     = 512

# ── Indic script Unicode ranges ───────────────────────────────────────────────
INDIC_RANGES = [
    (0x0900, 0x097F), (0x0980, 0x09FF), (0x0A00, 0x0A7F),
    (0x0A80, 0x0AFF), (0x0B00, 0x0B7F), (0x0B80, 0x0BFF),
    (0x0C00, 0x0C7F), (0x0C80, 0x0CFF), (0x0D00, 0x0D7F),
]
SUPPORTED_IMG = {".png", ".jpg", ".jpeg", ".tiff", ".tif", ".bmp", ".webp"}

print("✅ Config ready")


Running on: CUDA
  GPU 0: Tesla T4  |  VRAM: 15.6 GB
  GPU 1: Tesla T4  |  VRAM: 15.6 GB
✅ Config ready


In [2]:
class ModelRegistry:
    """
    Single container for all loaded models.
    Load once at the top of the session — never reload between pipeline calls.
    """
    def __init__(self):
        self.gen_model         = None   # Qwen2.5-7B-Instruct (causal LM)
        self.gen_tokenizer     = None
        self.embed_model       = None   # BAAI/bge-base-en-v1.5
        self.embed_tokenizer   = None
        self.rerank_model      = None   # cross-encoder TinyBERT
        self.rerank_tokenizer  = None
        self.colpali_model     = None   # ColPali v1.2 page encoder
        self.colpali_processor = None
        self.vlm_model         = None   # Qwen2.5-VL-7B vision-language
        self.vlm_processor     = None
        self._loaded_flags     = {}

    def status(self):
        parts = []
        for name in ["gen","embed","rerank","colpali","vlm"]:
            model_attr = f"{name}_model"
            loaded = getattr(self, model_attr) is not None
            parts.append(f"{'✅' if loaded else '❌'} {name}")
        print("  |  ".join(parts))

print("✅ ModelRegistry defined")


✅ ModelRegistry defined


In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    AutoProcessor,
    BitsAndBytesConfig,
)


# 4-bit NF4 config — keeps large models in ~8 GB VRAM on a T4
BNB_4BIT = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_use_double_quant = True,
    bnb_4bit_quant_type       = "nf4",
    bnb_4bit_compute_dtype    = torch.float16,
)

M = ModelRegistry()   # <── global registry, referenced by ALL pipeline cells

# ── 1. Text generator (Qwen2.5-7B-Instruct) ──────────────────────────────────
print("Loading generator …")
M.gen_tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_NAME, trust_remote_code=True)
M.gen_model = AutoModelForCausalLM.from_pretrained(
    GEN_MODEL_NAME,
    trust_remote_code   = True,
    torch_dtype         = torch.float16,
    device_map          = "auto",
    quantization_config = BNB_4BIT,
).eval()
print("  ✅ Generator loaded")

# ── 2. BGE embedder (stays on CPU — fast & saves VRAM) ───────────────────────
print("Loading embedder …")
M.embed_tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL_NAME)
M.embed_model     = AutoModel.from_pretrained(EMBED_MODEL_NAME).eval().to("cpu")
print("  ✅ Embedder loaded (CPU)")

# ── 3. Cross-encoder reranker ─────────────────────────────────────────────────
print("Loading reranker …")
M.rerank_tokenizer = AutoTokenizer.from_pretrained(RERANK_MODEL_NAME)
M.rerank_model     = AutoModelForSequenceClassification.from_pretrained(
    RERANK_MODEL_NAME
).eval()
print("  ✅ Reranker loaded")

# ── 4. ColPali v1.2 page encoder ─────────────────────────────────────────────
print("Loading ColPali …")
from colpali_engine.models import ColPali, ColPaliProcessor
M.colpali_model     = ColPali.from_pretrained(
    COLPALI_ID, torch_dtype=torch.float16, device_map={"": DEVICE}
).eval()
M.colpali_processor = ColPaliProcessor.from_pretrained(COLPALI_ID)
print("  ✅ ColPali loaded")

# ── 5. Qwen2.5-VL-7B vision-language model ───────────────────────────────────
print("Loading VLM (Qwen2.5-VL-7B) …")
try:
    from transformers import Qwen2_5_VLForConditionalGeneration as _QwenVL
except ImportError:
    from transformers import Qwen2VLForConditionalGeneration as _QwenVL

M.vlm_processor = AutoProcessor.from_pretrained(
    QWEN_VL_ID,
    min_pixels = 256  * 28 * 28,
    max_pixels = 1280 * 28 * 28,
)
M.vlm_model = _QwenVL.from_pretrained(
    QWEN_VL_ID,
    torch_dtype         = torch.float16,
    device_map          = "auto",
    trust_remote_code   = True,
    quantization_config = BNB_4BIT,
).eval()
print("  ✅ VLM loaded")

if DEVICE == "cuda":
    used  = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"\nVRAM used: {used:.1f} / {total:.1f} GB")

print("\n🎉 All models ready — registry object: M")
M.status()


2026-05-21 05:25:44.518340: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779341144.764147     260 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779341144.834611     260 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779341145.408925     260 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779341145.408983     260 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779341145.408986     260 computation_placer.cc:177] computation placer alr

Loading generator …


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model-00002-of-00005.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00003-of-00005.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00004-of-00005.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00005-of-00005.safetensors:   0%|          | 0.00/1.09G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

## 🛠️ Shared Text Utilities

In [ ]:
import re, sqlite3, json
import torch
import torch.nn.functional as F
import numpy as np
from typing import List, Optional, Dict, Any, Tuple, TypedDict

# ── Token helpers ─────────────────────────────────────────────────────────────
def count_tokens(text: str, tokenizer) -> int:
    if not text or not text.strip():
        return 0
    return len(tokenizer.encode(text.strip(), add_special_tokens=False))

def split_sentences_robust(text: str) -> List[str]:
    if not text:
        return []
    protected = re.sub(
        r"\b(Dr|Mr|Mrs|Ms|Prof|Sr|Jr|vs|etc|Inc|Ltd|Corp|Co|Rs|com|in)\.\s",
        r"\1<PERIOD> ", text,
    )
    sentences = re.split(r"(?<=[.!?])\s+(?=[A-Z])", protected)
    return [s.replace("<PERIOD>", ".").strip() for s in sentences if s.strip()]

def truncate_at_sentence(text, tokenizer, max_tokens, add_ellipsis=True):
    if not text:
        return ""
    if count_tokens(text, tokenizer) <= max_tokens:
        return text
    sentences = split_sentences_robust(text)
    acc, acc_t = [], 0
    for s in sentences:
        t = count_tokens(s, tokenizer)
        if acc_t + t <= max_tokens:
            acc.append(s); acc_t += t
        else:
            break
    if acc:
        out = " ".join(acc)
        return out + ("..." if add_ellipsis else "")
    tokens = tokenizer.encode(text, add_special_tokens=False)[:max_tokens]
    return tokenizer.decode(tokens, skip_special_tokens=True) + ("..." if add_ellipsis else "")

def pack_chunks_to_max_tokens(chunks, tokenizer, max_tokens):
    packed, used = [], 0
    for c in chunks:
        text = c.get("text", "").strip()
        if not text:
            continue
        tc = count_tokens(text, tokenizer)
        if used + tc <= max_tokens:
            packed.append(text); used += tc
        else:
            rem = max_tokens - used
            if rem < 40:
                break
            tr = truncate_at_sentence(text, tokenizer, rem)
            if tr:
                tr_tc = count_tokens(tr, tokenizer)
                if used + tr_tc <= max_tokens:
                    packed.append(tr)
            break
    return "\n\n".join(packed)

def sanitize_output(text):
    if not text: return ""
    text = re.sub(r"```.*?```", "", text, flags=re.S)
    text = re.sub(r"\b(Human|Assistant):.*", "", text, flags=re.I)
    return re.sub(r"\n{2,}", "\n\n", text).strip()

def extract_final_answer(text):
    if not text: return ""
    t = text.strip()
    if "###" in t: t = t.split("###")[-1]
    t = t.replace("Human:", "").replace("Assistant:", "").strip()
    t = re.sub(r"(?i)^answer\s*[:\-]?", "", t).strip()
    t = re.sub(r"\n{2,}", " ", t).strip()
    sentences = re.split(r"(?<=[.!?])\s+", t)
    if not sentences: return ""
    if not re.search(r"[.!?]$", sentences[-1]):
        sentences = sentences[:-1]
    return " ".join(sentences).strip()

def generate_with_llm(prompt, is_greeting=False):
    """Core LLM call — uses global M registry."""
    max_tokens = 15 if is_greeting else 300
    messages   = [{"role": "user", "content": prompt}]
    inputs     = M.gen_tokenizer.apply_chat_template(
        messages, return_tensors="pt", add_generation_prompt=True
    ).to(M.gen_model.device)
    with torch.no_grad():
        out = M.gen_model.generate(
            inputs, max_new_tokens=max_tokens,
            temperature=0.2, do_sample=False,
            repetition_penalty=1.2,
            eos_token_id=M.gen_tokenizer.eos_token_id,
            pad_token_id=M.gen_tokenizer.eos_token_id,
        )
    return M.gen_tokenizer.decode(
        out[0][inputs.shape[-1]:], skip_special_tokens=True
    ).strip()
def embed_texts(texts: list, is_query: bool = False) -> np.ndarray:
    """
    Encode a list of strings using the BGE embedder.
    Works for both passage encoding (is_query=False) and query encoding (is_query=True).
    Returns a float32 numpy array of shape (len(texts), hidden_dim).
    """
    if not texts:
        return np.empty((0, 768), dtype=np.float32)

    # BGE prepends a task prefix for queries
    prefix = "Represent this sentence for searching relevant passages: " if is_query else ""
    prefixed = [prefix + t for t in texts]

    BATCH = 32
    all_embs = []

    for i in range(0, len(prefixed), BATCH):
        batch = prefixed[i : i + BATCH]
        enc = M.embed_tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt",
        ).to(M.embed_model.device)

        with torch.no_grad():
            out = M.embed_model(**enc)

        # CLS-token pooling
        emb = out.last_hidden_state[:, 0, :]
        emb = F.normalize(emb, dim=-1)
        all_embs.append(emb.cpu().numpy())

    return np.vstack(all_embs).astype(np.float32)

print("✅ embed_texts defined")
print("✅ Shared utilities ready")


In [ ]:
"""
file_store — Out-of-state file registry for LangGraph pipelines.
Shared across all cells via sys.modules["file_store"].
"""

import threading, pathlib, sys
from typing import Dict, Optional
import pandas as pd

_store: Dict[str, Dict[str, dict]] = {}
_lock  = threading.Lock()


def register_files(thread_id: str, session_files: dict) -> None:
    with _lock:
        existing = _store.get(thread_id, {})
        for fname, rec in session_files.items():
            existing[fname] = {
                "path": rec.get("path", ""),
                "text": rec.get("text", "") or "",
                "df"  : rec.get("df"),
            }
        _store[thread_id] = existing

def clear_files(thread_id: str) -> None:
    with _lock:
        _store.pop(thread_id, None)

def get_df(thread_id: str, fname: str) -> Optional[pd.DataFrame]:
    with _lock:
        return _store.get(thread_id, {}).get(fname, {}).get("df")

def get_pdf_text(thread_id: str, fname: str) -> str:
    with _lock:
        return _store.get(thread_id, {}).get(fname, {}).get("text", "")

def get_path(thread_id: str, fname: str) -> str:
    with _lock:
        return _store.get(thread_id, {}).get(fname, {}).get("path", "")

def get_all_dfs(thread_id: str) -> Dict[str, pd.DataFrame]:
    with _lock:
        files = _store.get(thread_id, {})
        return {f: r["df"] for f, r in files.items() if r.get("df") is not None}

def get_all_pdf_texts(thread_id: str) -> Dict[str, str]:
    with _lock:
        files = _store.get(thread_id, {})
        return {f: r["text"] for f, r in files.items() if r.get("text")}

def get_all_paths(thread_id: str) -> Dict[str, str]:
    with _lock:
        return {f: r["path"] for f, r in _store.get(thread_id, {}).items()}

def get_combined_pdf_text(thread_id: str, max_chars_per_file: int = 8000) -> str:
    parts = []
    for fname, text in get_all_pdf_texts(thread_id).items():
        parts.append(f"=== {fname} ===\n{text[:max_chars_per_file]}")
    return "\n\n".join(parts)

def list_files(thread_id: str) -> list:
    with _lock:
        return list(_store.get(thread_id, {}).keys())


# ── Write to disk so `from file_store import ...` works in all cells ──────────
_DEST = pathlib.Path("/kaggle/working/file_store.py")
_module_src = """
import threading
from typing import Dict, Optional
import pandas as pd

_store: Dict[str, Dict[str, dict]] = {}
_lock  = threading.Lock()

def register_files(thread_id, session_files):
    with _lock:
        existing = _store.get(thread_id, {})
        for fname, rec in session_files.items():
            existing[fname] = {
                "path": rec.get("path", ""),
                "text": rec.get("text", "") or "",
                "df"  : rec.get("df"),
            }
        _store[thread_id] = existing

def clear_files(thread_id):
    with _lock:
        _store.pop(thread_id, None)

def get_df(thread_id, fname):
    with _lock:
        return _store.get(thread_id, {}).get(fname, {}).get("df")

def get_pdf_text(thread_id, fname):
    with _lock:
        return _store.get(thread_id, {}).get(fname, {}).get("text", "")

def get_path(thread_id, fname):
    with _lock:
        return _store.get(thread_id, {}).get(fname, {}).get("path", "")

def get_all_dfs(thread_id):
    with _lock:
        files = _store.get(thread_id, {})
        return {f: r["df"] for f, r in files.items() if r.get("df") is not None}

def get_all_pdf_texts(thread_id):
    with _lock:
        files = _store.get(thread_id, {})
        return {f: r["text"] for f, r in files.items() if r.get("text")}

def get_all_paths(thread_id):
    with _lock:
        return {f: r["path"] for f, r in _store.get(thread_id, {}).items()}

def get_combined_pdf_text(thread_id, max_chars_per_file=8000):
    parts = []
    for fname, text in get_all_pdf_texts(thread_id).items():
        parts.append(f"=== {fname} ===\\n{text[:max_chars_per_file]}")
    return "\\n\\n".join(parts)

def list_files(thread_id):
    with _lock:
        return list(_store.get(thread_id, {}).keys())
"""
_DEST.write_text(_module_src)

# Register this notebook module as "file_store" so all imports share the same _store
sys.modules["file_store"] = sys.modules[__name__]
print(f"✅ file_store written to {_DEST}")
print("✅ file_store registered in sys.modules — all cells share the same _store")


## 📄 Text RAG Pipeline (PyMuPDF + FAISS + BM25 + Cross-Encoder + MMR + dynamic_trim)

In [ ]:
import hashlib, re, faiss
import numpy as np
import torch
import torch.nn.functional as F
from rank_bm25 import BM25Okapi
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

# ─────────────────────────────────────────────────────────────────────────────
#  Text helpers (keep the ones already in shared utilities; these extend them)
# ─────────────────────────────────────────────────────────────────────────────

def is_heading(line: str) -> bool:
    line = line.strip()
    if not line: return False
    if line.startswith(("•", "-", "*")): return False
    if len(line.split()) > 10: return False
    if re.match(r'^\d{1,3}[\.\-\s]+[A-Za-z]', line): return True
    if re.match(r'^#\d{1,3}\s+[A-Za-z]', line): return True
    if line.isupper() and 3 <= len(line.split()) <= 8: return True
    return False


def split_into_sections(pages: list) -> list:
    sections, cur_title, cur_text = [], None, []
    for page in pages:
        page_text = page.get("text", "") if isinstance(page, dict) else page
        if not page_text or not page_text.strip(): continue
        for raw_line in page_text.splitlines():
            line = raw_line.strip()
            if not line: continue
            if is_heading(line):
                if cur_title and cur_text:
                    sections.append({"section_title": cur_title,
                                      "text": "\n".join(cur_text).strip()})
                cur_title, cur_text = line, []
            else:
                cur_text.append(line)
    if cur_title and cur_text:
        sections.append({"section_title": cur_title,
                          "text": "\n".join(cur_text).strip()})
    return sections


def extract_metadata(title: str, text: str) -> dict:
    tl = title.lower()
    cat = "general"
    if any(k in tl for k in ["about", "management", "financial"]):
        cat = "company_info"
    elif any(k in tl for k in ["service", "alert", "consultancy", "kpo"]):
        cat = "service"
    elif any(k in tl for k in ["portal", "platform", "procure", "auction"]):
        cat = "product"
    elif any(k in tl for k in ["classification", "ai", "methodology", "token"]):
        cat = "technical"
    m = re.match(r'^(\d+)\s+', title)
    return {"section_number": int(m.group(1)) if m else None, "category": cat}


def split_large_section(section: dict, max_tokens: int = 480,
                         overlap_ratio: float = 0.10) -> list:
    text, title = section["text"].strip(), section["section_title"]
    if count_tokens(text, M.embed_tokenizer) <= max_tokens:
        return [{"section_title": title, "chunk_index": 0, "text": text,
                 "source": "document",
                 "token_count": count_tokens(text, M.embed_tokenizer)}]
    sents = split_sentences_robust(text)
    chunks, cur, cur_t, idx = [], [], 0, 0
    overlap_budget = int(max_tokens * overlap_ratio)
    for sent in sents:
        st = count_tokens(sent, M.embed_tokenizer)
        if st > max_tokens:  # single mega-sentence → own chunk
            if cur:
                chunks.append({"section_title": title, "chunk_index": idx,
                                "text": " ".join(cur), "source": "document",
                                "token_count": cur_t})
                idx += 1
            chunks.append({"section_title": title, "chunk_index": idx,
                            "text": sent, "source": "document", "token_count": st})
            idx += 1; cur, cur_t = [], 0; continue
        if cur_t + st <= max_tokens:
            cur.append(sent); cur_t += st
        else:
            chunks.append({"section_title": title, "chunk_index": idx,
                            "text": " ".join(cur), "source": "document",
                            "token_count": cur_t})
            idx += 1
            # overlap carry-back
            ov, ovt = [], 0
            for prev in reversed(cur):
                pt = count_tokens(prev, M.embed_tokenizer)
                if ovt + pt <= overlap_budget: ov.insert(0, prev); ovt += pt
                else: break
            cur, cur_t = ov + [sent], ovt + st
        if idx >= 20: break
    if cur and idx < 20:
        chunks.append({"section_title": title, "chunk_index": idx,
                        "text": " ".join(cur), "source": "document",
                        "token_count": cur_t})
    return chunks


def semantic_subdivide(section: dict, max_tokens: int = 350) -> list:
    text, title = section["text"].strip(), section["section_title"]
    # Bullet lists → split on bullet boundary first
    if "•" in text or re.search(r"^[-\*] ", text, re.M):
        bullets = [b.strip() for b in re.split(r"[•\-\*]\s+", text) if b.strip()]
        if len(bullets) >= 3:
            return [{"section_title": title, "text": b} for b in bullets]
    if count_tokens(text, M.embed_tokenizer) <= max_tokens:
        return [section]
    # Paragraph split
    paras = [p.strip() for p in text.split("\n\n") if p.strip()]
    sub, cur, cur_t = [], [], 0
    for p in paras:
        pt = count_tokens(p, M.embed_tokenizer)
        if cur_t + pt <= max_tokens:
            cur.append(p); cur_t += pt
        else:
            if cur: sub.append({"section_title": title, "text": "\n\n".join(cur)})
            cur, cur_t = [p], pt
    if cur: sub.append({"section_title": title, "text": "\n\n".join(cur)})
    return sub or [section]


def _normalize_text(t: str) -> str:
    return re.sub(r"\s+", " ", t.lower()).strip()


def deduplicate_chunks(chunks: list) -> list:
    seen, unique = set(), []
    for c in chunks:
        h = hashlib.md5(_normalize_text(c["text"]).encode()).hexdigest()
        if h not in seen:
            seen.add(h); unique.append(c)
    return unique


# ─────────────────────────────────────────────────────────────────────────────
#  build_rag_index_pymupdf — upgraded replacement for build_rag_index
#  Uses PyMuPDF (better text extraction) + dedup + overlap chunking
# ─────────────────────────────────────────────────────────────────────────────
def _bm25_tokenize(text: str) -> list:
    return re.findall(r"\w+", text.lower())



# ── build_greeting_embeddings — required by build_rag_index_pymupdf ───────────
# FIX: was called but never defined in original notebook.
def build_greeting_embeddings() -> dict:
    """
    Pre-compute embeddings for common greeting phrases.
    Stored in the RAG index and used by the router for fast greeting detection.
    Returns {phrase: np.ndarray} or {} on error.
    """
    GREETINGS = [
        "hello", "hi", "hey", "good morning", "good evening",
        "good afternoon", "howdy", "greetings", "hi there", "hello there",
    ]
    try:
        embs = embed_texts(GREETINGS, is_query=False)
        return {phrase: embs[i] for i, phrase in enumerate(GREETINGS)}
    except Exception as e:
        print(f"  ⚠️  build_greeting_embeddings skipped: {e}")
        return {}

def build_rag_index_pymupdf(doc_path: str,
                             min_tokens: int = 50,
                             hard_max: int = 480,
                             overlap_ratio: float = 0.10) -> dict:
    """
    Upgraded RAG indexer — works for PDFs, DOCX, TXT.
    Drop-in replacement for the original build_rag_index().
    """
    p   = Path(doc_path)
    ext = p.suffix.lower()
    print(f"[RAG] Indexing: {p.name}")

    # ── Load pages ────────────────────────────────────────────────────────────
    if ext == ".pdf":
        pages = load_pdf_pages_pymupdf(doc_path)         # PyMuPDF
    elif ext in (".docx", ".doc"):
        import docx as _docx
        d = _docx.Document(doc_path)
        full_text = "\n".join(para.text for para in d.paragraphs)
        pages = [{"page": 1, "text": full_text}]
    elif ext in (".txt", ".md"):
        pages = [{"page": 1, "text": p.read_text(errors="ignore")}]
    else:
        raise ValueError(f"Unsupported doc type for RAG: {ext}")

    print(f"  Pages loaded: {len(pages)}")

    # ── Section + chunk ───────────────────────────────────────────────────────
    sections = split_into_sections(pages)
    pre_chunks = []
    for sec in sections:
        pre_chunks.extend(semantic_subdivide(sec, max_tokens=350))

    final_chunks = []
    for sec in pre_chunks:
        title, text = sec["section_title"], sec["text"]
        tc   = count_tokens(text, M.embed_tokenizer)
        meta = extract_metadata(title, text)
        sem  = f"{title}. {text}".strip()
        if tc <= hard_max:
            final_chunks.append({"section_title": title, "chunk_index": 0,
                                  "text": text, "semantic_text": sem,
                                  "source": "document", "token_count": tc, **meta})
        else:
            for chunk in split_large_section(sec, hard_max, overlap_ratio):
                chunk["token_count"]   = count_tokens(chunk["text"], M.embed_tokenizer)
                chunk["semantic_text"] = f"{title}. {chunk['text']}".strip()
                chunk.update(meta)
                final_chunks.append(chunk)

    final_chunks = deduplicate_chunks(
        [c for c in final_chunks if c["token_count"] >= min_tokens]
    )
    print(f"  Chunks after dedup: {len(final_chunks)}")

    # ── Embed + FAISS ─────────────────────────────────────────────────────────
    texts      = [c["semantic_text"] for c in final_chunks]
    embeddings = embed_texts(texts, is_query=False)
    dim        = embeddings.shape[1]
    idx        = faiss.IndexFlatIP(dim)
    idx.add(embeddings)
    bm25       = BM25Okapi([_bm25_tokenize(t) for t in texts])
    meta_list  = [{"chunk_id": i, "chunk_index": c.get("chunk_index", i),
                   "section_title": c["section_title"], "text": c["text"],
                   "source": c.get("source", "document"),
                   "category": c.get("category"),
                   "section_number": c.get("section_number")}
                  for i, c in enumerate(final_chunks)]
    greet_embs = build_greeting_embeddings()
    print("[RAG] ✅ Index ready")
    return {"faiss_index": idx, "bm25_index": bm25, "chunk_metadata": meta_list,
            "chunks": final_chunks, "greeting_embeddings": greet_embs}


# ─────────────────────────────────────────────────────────────────────────────
#  dynamic_trim — ported from working chatbot
#  Removes hallucinated / drifted sentences from LLM output
# ─────────────────────────────────────────────────────────────────────────────
def dynamic_trim(answer: str, chunks: list,
                 strong_th: float = 0.55, weak_th: float = 0.30,
                 patience: int = 2):
    """Filter LLM output sentences by semantic similarity to retrieved chunks."""
    if not answer or not answer.strip():
        return None
    context_texts = [c["text"] for c in chunks if c.get("text")]
    if not context_texts:
        return answer
    context_embs = embed_texts(context_texts, is_query=False)
    context_emb  = np.mean(context_embs, axis=0, keepdims=True)
    units = [u.strip() for u in answer.split("\n") if u.strip()]
    kept, drift_count = [], 0
    for unit in units:
        unit_emb = embed_texts([unit], is_query=True)
        sim      = float(np.dot(unit_emb, context_emb.T))
        if sim >= strong_th:
            kept.append(unit); drift_count = 0
        elif sim >= weak_th:
            kept.append(unit); drift_count += 1
        else:
            drift_count += 1
            if drift_count >= patience: break
    return "\n".join(kept).strip() if kept else None


# ─────────────────────────────────────────────────────────────────────────────
#  fallback_llm — targeted single-chunk fallback from working chatbot
# ─────────────────────────────────────────────────────────────────────────────
def fallback_llm(chunk_text: str, query: str) -> str:
    prompt = (
        "You are a strict document support assistant.\n"
        "Use ONLY the documentation below to answer.\n"
        "Keep it short (1–2 sentences) unless asked for detail.\n"
        "If not found, say exactly:\n"
        "\"Sorry. This information is not available in the current documentation.\"\n\n"
        f"Documentation:\n{chunk_text}\n\nQuestion:\n{query}"
    )
    return generate_with_llm(prompt, is_greeting=False)


# ─────────────────────────────────────────────────────────────────────────────
#  Retrieval helpers (from original IntelliDoc — unchanged interface)
# ─────────────────────────────────────────────────────────────────────────────
def _kw_score(query: str, text: str) -> float:
    qt = set(re.findall(r"\w+", query.lower()))
    tt = set(re.findall(r"\w+", text.lower()))
    return len(qt & tt) / len(qt) if qt else 0.0


def faiss_retrieve(query: str, rag_index: dict, top_k: int = 15,
                   alpha: float = 0.15) -> list:
    q_emb      = embed_texts([query], is_query=True)
    scores, ii = rag_index["faiss_index"].search(q_emb, top_k)
    meta       = rag_index["chunk_metadata"]
    results    = []
    for sc, idx in zip(scores[0], ii[0]):
        m  = meta[idx]
        kw = _kw_score(query, m.get("text", ""))
        results.append({**m, "score": float(sc), "keyword_score": kw,
                         "hybrid_score": float(sc) + alpha * kw})
    return sorted(results, key=lambda x: x["hybrid_score"], reverse=True)


def bm25_retrieve(query: str, rag_index: dict, top_k: int = 10) -> list:
    scores = rag_index["bm25_index"].get_scores(_bm25_tokenize(query))
    ranked = np.argsort(scores)[::-1][:top_k]
    meta   = rag_index["chunk_metadata"]
    return [{**meta[i], "bm25_score": float(scores[i])}
            for i in ranked if scores[i] > 0]


def cross_encoder_rerank(query: str, chunks: list, top_n: int = 5,
                          max_length: int = 512) -> list:
    if not chunks: return []
    pairs  = [(query, c["text"]) for c in chunks]
    inputs = M.rerank_tokenizer(pairs, padding=True, truncation=True,
                                max_length=max_length, return_tensors="pt")
    inputs = {k: v.to(M.rerank_model.device) for k, v in inputs.items()}
    with torch.no_grad():
        scores = M.rerank_model(**inputs).logits.squeeze(-1).cpu().tolist()
    reranked = sorted([{**c, "rerank_score": float(s)}
                        for s, c in zip(scores, chunks)],
                       key=lambda x: x["rerank_score"], reverse=True)
    return reranked[:top_n]


def mmr_select(chunks: list, k: int = 4, lambda_param: float = 0.6) -> list:
    if len(chunks) <= k: return chunks
    texts = [c["text"] for c in chunks]
    embs  = embed_texts(texts, is_query=False)
    rel   = [c.get("rerank_score") or c.get("hybrid_score") or c.get("score", 0.0)
             for c in chunks]
    sel, rem = [], list(range(len(chunks)))
    for _ in range(k):
        if not rem: break
        if not sel:
            best = max(rem, key=lambda i: rel[i])
        else:
            sel_e = np.stack([embs[i] for i in sel])
            best, bmv = rem[0], -1e9
            for i in rem:
                mmr_v = lambda_param * rel[i] - (1 - lambda_param) * float(
                    np.max(embs[i] @ sel_e.T))
                if mmr_v > bmv: bmv, best = mmr_v, i
        sel.append(best); rem.remove(best)
    return [chunks[i] for i in sel]


def retrieve_and_rank(query: str, rag_index: dict, top_k: int = 10) -> dict:
    sem    = faiss_retrieve(query, rag_index, top_k=top_k)
    lex    = bm25_retrieve(query, rag_index, top_k=top_k)
    seen, merged = set(), []
    for c in sem + lex:
        key = c.get("chunk_index")
        if key not in seen: seen.add(key); merged.append(c)
    if len(merged) > 1:
        merged = cross_encoder_rerank(query, merged, top_n=8)
    if len(merged) > 3:
        merged = mmr_select(merged, k=4)
    rag_score = 0.0
    if merged:
        rag_score = (merged[0].get("rerank_score") or
                     merged[0].get("hybrid_score") or
                     merged[0].get("score", 0.0))
    context = pack_chunks_to_max_tokens(merged, M.gen_tokenizer, CONTEXT_BUDGET)
    return {"ranked_chunks": merged, "rag_score": rag_score, "context_text": context}


def rag_answer(query: str, rag_index: dict, history: list = None) -> dict:
    """
    Full RAG answer with dynamic_trim + fallback_llm.
    Upgraded from original IntelliDoc.
    """
    history  = history or []
    res      = retrieve_and_rank(query, rag_index)
    chunks, rag_score, context = (res["ranked_chunks"],
                                   res["rag_score"],
                                   res["context_text"])

    if not chunks:
        return {"answer": "Sorry. This information is not available in the documentation.",
                "rag_score": 0.0, "chunks": [], "grounded": False}

    hist_txt = "\n".join(f"- {t['user']}" for t in history[-3:]) if history else ""
    prompt = (
        "<|system|>\n"
        "You are a detailed document support assistant.\n"
        "Rules:\n"
        "- Use ONLY the information explicitly available in the documentation below.\n"
        "- Provide detailed and well-structured answers.\n"
        "- If the answer contains multiple points, explain them in bullet points.\n"
        "- If the documentation includes lists, steps, features, benefits, or categories, preserve them in list format.\n"
        "- If the answer requires explanation, elaborate clearly with proper formatting.\n"
        "- If query has a greeting, start with a short friendly acknowledgement.\n"
        "- Do NOT make up information outside the provided documentation.\n"
        "- If answer is not found, say exactly:\n"
        "  \"Sorry. This information is not available in the documentation.\"\n\n"
        f"<|user|>\nDocumentation:\n{context}\n\n"
        f"{f'Previous messages:\\n{hist_txt}\\n\\n' if hist_txt else ''}"
        f"Question:\n{query}\n\n"
        "<|assistant|>\n"
    )
    raw_answer = generate_with_llm(prompt, is_greeting=False)
    answer     = dynamic_trim(raw_answer, chunks)

    HIGH, LOW = 0.55, 0.30
    if answer and rag_score >= LOW:
        final    = answer
        grounded = rag_score >= HIGH
    elif rag_score < LOW or not answer:
        final    = fallback_llm(chunks[0]["text"], query)
        grounded = False
    else:
        final    = "Sorry. This information is not available in the documentation."
        grounded = False

    return {"answer": final.strip(), "rag_score": rag_score,
            "chunks": chunks, "grounded": grounded}


print("✅ Upgraded Text RAG pipeline ready (PyMuPDF + dedup + dynamic_trim + fallback)")

## 🗄️ SQL RAG Pipeline (Schema introspection · Column embedding · LLM SQL gen · Plotly viz)

In [ ]:
import sqlite3, json, pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display
import io, base64

# Global SQL database connection (in-memory, persists for session)
SQL_CONN  = sqlite3.connect(":memory:", check_same_thread=False)

# Per-table column-value embeddings for non-numeric columns
_col_value_embeddings: Dict[str, Dict[str, Any]] = {}

# ── Load Excel/CSV into SQLite ────────────────────────────────────────────────
def load_file_to_sql(file_path: str, table_name: str = None, embed_cols: bool = True) -> str:
    path = Path(file_path)
    ext  = path.suffix.lower()
    if ext in (".xlsx", ".xlsm", ".xls"):
        dfs = pd.read_excel(file_path, sheet_name=None)
        loaded_tables = []
        for sheet, df in dfs.items():
            tname = table_name or re.sub(r"\W+","_", sheet.lower())
            df.columns = [re.sub(r"\W+","_",c.strip().lower()) for c in df.columns]
            df.to_sql(tname, SQL_CONN, if_exists="replace", index=False)
            loaded_tables.append(tname)
            if embed_cols:
                try:
                    embed_text_columns(tname, df)
                except Exception as e:
                    print(f"  ⚠️  embed_text_columns skipped: {e}")
            print(f"  ✅ Sheet '{sheet}' → table '{tname}' ({len(df)} rows)")
        return loaded_tables
    elif ext == ".csv":
        df = None
        for enc in ("utf-8", "latin-1", "cp1252", "iso-8859-1"):
            try:
                df = pd.read_csv(file_path, encoding=enc)
                break
            except UnicodeDecodeError:
                continue
        if df is None:
            df = pd.read_csv(file_path, encoding="utf-8", errors="replace")
        tname = table_name or re.sub(r"\W+","_", path.stem.lower())
        df.columns = [re.sub(r"\W+","_",c.strip().lower()) for c in df.columns]
        df.to_sql(tname, SQL_CONN, if_exists="replace", index=False)
        if embed_cols:
            try:
                embed_text_columns(tname, df)
            except Exception as e:
                print(f"  ⚠️  embed_text_columns skipped: {e}")
        print(f"  ✅ CSV → table '{tname}' ({len(df)} rows)")
        return [tname]
    else:
        raise ValueError(f"Unsupported file type: {ext}")


# ── Schema introspection ──────────────────────────────────────────────────────
def get_schema_for_llm(table_names: List[str] = None) -> str:
    """
    Return a rich schema string for all (or specified) tables, including:
    - column name + SQLite type
    - pandas inferred dtype
    - sample distinct values for text/categorical columns (max 10)
    This is injected verbatim into the SQL-generation prompt.
    """
    cursor = SQL_CONN.cursor()
    if table_names is None:
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
        table_names = [r[0] for r in cursor.fetchall()]

    lines = []
    for tname in table_names:
        cursor.execute(f"PRAGMA table_info('{tname}')")
        cols   = cursor.fetchall()          # (cid, name, type, notnull, dflt, pk)
        cursor.execute(f"SELECT COUNT(*) FROM '{tname}'")
        nrows  = cursor.fetchone()[0]
        lines.append(f"TABLE: {tname}  ({nrows} rows)")

        df_sample = pd.read_sql(f"SELECT * FROM '{tname}' LIMIT 200", SQL_CONN)

        for _, name, sqlite_type, notnull, _, pk in cols:
            pd_dtype = str(df_sample[name].dtype) if name in df_sample else "unknown"
            nullable = "NOT NULL" if notnull else "nullable"
            pk_flag  = " [PK]" if pk else ""
            base     = f"  {name}  {sqlite_type}  (pandas: {pd_dtype})  {nullable}{pk_flag}"

            is_numeric = pd.api.types.is_numeric_dtype(df_sample[name]) if name in df_sample else False
            if not is_numeric and name in df_sample:
                vals = df_sample[name].dropna().astype(str).unique()[:10].tolist()
                if vals:
                    base += f"  | samples: {vals}"
            lines.append(base)
        lines.append("")

    return "\n".join(lines)


# ── Embed non-numeric column values ───────────────────────────────────────────
def embed_text_columns(table_name: str, df: pd.DataFrame):
    _col_value_embeddings.setdefault(table_name, {})
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            continue
        unique_vals = df[col].dropna().astype(str).unique().tolist()
        if not unique_vals or len(unique_vals) > 500:
            continue
        embs = embed_texts(unique_vals, is_query=False)
        _col_value_embeddings[table_name][col] = {
            v: embs[i] for i, v in enumerate(unique_vals)
        }
    print(f"  🔍 Embedded text columns for table '{table_name}'")


def semantic_value_lookup(table_name, col_name, query_term, top_k=3) -> List[str]:
    store = _col_value_embeddings.get(table_name, {}).get(col_name)
    if not store:
        return []
    q_emb  = embed_texts([query_term], is_query=True)[0]
    scores = {v: float(np.dot(q_emb, e)) for v, e in store.items()}
    return sorted(scores, key=scores.get, reverse=True)[:top_k]


# ── SQL generation + execution ────────────────────────────────────────────────
def _parse_sql_json(raw: str) -> dict:
    """
    Parse LLM JSON response for SQL generation.
    Expected shape:
        {
            "sql":       "SELECT ...",
            "confident": true,
            "reasoning": "brief explanation of approach"
        }
    Returns dict with keys: sql, confident, reasoning, parse_error.
    """
    # Strip markdown fences if LLM wrapped in ```json ... ```
    cleaned = re.sub(r"```json|```", "", raw, flags=re.I).strip()
    try:
        parsed = json.loads(cleaned)
    except json.JSONDecodeError:
        # Fallback: try to extract a JSON object via regex
        m = re.search(r"\{.*\}", cleaned, re.S)
        if m:
            try:
                parsed = json.loads(m.group())
            except json.JSONDecodeError:
                return {"sql": None, "confident": False,
                        "reasoning": "", "parse_error": f"JSON decode failed: {raw[:200]}"}
        else:
            return {"sql": None, "confident": False,
                    "reasoning": "", "parse_error": f"No JSON found in output: {raw[:200]}"}

    sql = parsed.get("sql", "").strip()
    # Strip trailing semicolon inconsistency
    if ";" in sql:
        sql = sql[:sql.index(";")+1].strip()

    return {
        "sql"        : sql,
        "confident"  : bool(parsed.get("confident", True)),
        "reasoning"  : parsed.get("reasoning", ""),
        "parse_error": None,
    }


def _build_first_attempt_prompt(nl_query: str, schema: str, table_names: List[str]) -> str:
    """
    Prompt for attempt 1 — returns JSON with sql, confident flag, and reasoning.
    """
    tables_str = ", ".join(table_names) if table_names else "see schema below"
    return (
        "You are a precise SQLite query generator.\n\n"
        "RULES:\n"
        "1. Use column names EXACTLY as shown in the schema — same case, same spelling, same underscores.\n"
        "   Column names are sanitized: spaces→underscores, lowercased.\n"
        "   e.g. use `order_date` not `Order Date` or `OrderDate`.\n"
        "2. Use table names EXACTLY as listed under 'Available tables' — do NOT invent table names.\n"
        "3. For WHERE filter values, use sample_values from the schema — match case exactly.\n"
        "4. For numeric columns, validate range assumptions against min/max in the schema.\n"
        "5. For date columns stored as TEXT (dtype: object), check sample_values for the format:\n"
        "   - Format MM/DD/YYYY → strftime('%m', order_date)='09' AND strftime('%Y', order_date)='2014'\n"
        "   - Format YYYY-MM-DD → order_date LIKE '2014-09-%'\n"
        "   Always zero-pad month numbers: '09' not '9'.\n"
        "6. Only generate the query if you are confident it is correct and will execute without error.\n"
        "   If the question cannot be answered from the available schema, set confident=false and sql=null.\n\n"
        "OUTPUT FORMAT — respond with ONLY this JSON, no extra text:\n"
        "{\n"
        '  "sql":       "<the SQL query, or null if cannot be answered>",\n'
        '  "confident": <true if you are sure the query is correct, false otherwise>,\n'
        '  "reasoning": "<one sentence: what tables/columns/logic you used>"\n'
        "}\n\n"
        f"Available tables: {tables_str}\n\n"
        f"SCHEMA:\n{schema}\n\n"
        f"Question: {nl_query}\n\n"
        "JSON:"
    )


def _build_retry_prompt(nl_query: str, schema: str, table_names: List[str],
                        prev_sql: str, error_msg: str) -> str:
    """
    Prompt for retry — shows previous SQL + exact error, requests corrected JSON.
    """
    tables_str = ", ".join(table_names) if table_names else "see schema below"
    return (
        "You are a precise SQLite query generator. Your previous attempt produced an error.\n\n"
        "PREVIOUS ATTEMPT:\n"
        f"  SQL:   {prev_sql}\n"
        f"  Error: {error_msg}\n\n"
        "DIAGNOSE AND FIX — common causes:\n"
        "- Wrong table name → verify exactly against 'Available tables' below.\n"
        "- Wrong column name → verify spelling/underscores against SCHEMA below.\n"
        "- Date format mismatch → re-check sample_values for the actual stored format.\n"
        "- Filter value case mismatch → sample_values must be matched exactly.\n\n"
        "RULES:\n"
        "1. Column and table names exactly as in schema.\n"
        "2. Filter values must match sample_values case exactly.\n"
        "3. For TEXT date columns: use strftime('%m')='09' (zero-padded).\n"
        "4. Only produce a query you are confident will execute correctly.\n"
        "   If the question truly cannot be answered, set confident=false and sql=null.\n\n"
        "OUTPUT FORMAT — respond with ONLY this JSON, no extra text:\n"
        "{\n"
        '  "sql":       "<corrected SQL query, or null>",\n'
        '  "confident": <true if you are sure this will work, false otherwise>,\n'
        '  "reasoning": "<one sentence: what you changed and why>"\n'
        "}\n\n"
        f"Available tables: {tables_str}\n\n"
        f"SCHEMA:\n{schema}\n\n"
        f"Question: {nl_query}\n\n"
        "JSON:"
    )


def _build_error_rewrite_prompt(nl_query: str, schema: str, prev_sql: str,
                                 error_msg: str, reasoning: str = "") -> str:
    """
    Prompt to rewrite a final SQL failure into a helpful user-facing message.
    """
    reasoning_line = f"LLM reasoning: {reasoning}\n" if reasoning else ""
    return (
        "A user asked a data question. SQL generation failed after multiple attempts.\n"
        "Write a short, helpful response (2-3 sentences) that:\n"
        "1. Explains in plain English what went wrong (no raw SQL, no tracebacks).\n"
        "2. Tells the user what information IS available based on the schema.\n"
        "3. Suggests how they could rephrase their question.\n\n"
        f"User question: {nl_query}\n"
        f"Last SQL tried: {prev_sql or 'none generated'}\n"
        f"Error: {error_msg}\n"
        f"{reasoning_line}"
        f"\nAvailable schema summary:\n{schema}\n\n"
        "Helpful response:"
    )


def _build_data_to_text_prompt(nl_query: str, markdown_table: str) -> str:
    """
    Prompt to convert SQL result table into a natural language answer.
    """
    return (
        "You are a data analyst. A SQL query was run and returned the results below.\n"
        "Write a clear, concise answer (2-3 sentences) to the user's question using ONLY the data provided.\n"
        "If the result is a list, mention the count and a few examples.\n"
        "If the result is numeric, state the value with context.\n"
        "Do not mention SQL, tables, or technical details.\n\n"
        f"User question: {nl_query}\n\n"
        f"Query results:\n{markdown_table[:1500]}\n\n"
        "Answer:"
    )


def generate_sql_json(nl_query: str, table_names: List[str] = None,
                      error_msg: str = None, prev_sql: str = None) -> dict:
    """
    Call LLM and return parsed JSON dict:
        { sql, confident, reasoning, parse_error }
    Uses first-attempt prompt on attempt 1, retry prompt when error_msg+prev_sql present.
    """
    schema = get_schema_for_llm(table_names)
    if error_msg and prev_sql:
        prompt = _build_retry_prompt(nl_query, schema, table_names or [], prev_sql, error_msg)
    else:
        prompt = _build_first_attempt_prompt(nl_query, schema, table_names or [])
    raw    = generate_with_llm(prompt)
    result = _parse_sql_json(raw)
    print(f"  [LLM JSON] confident={result['confident']} | reasoning={result['reasoning'][:80]}")
    if result["parse_error"]:
        print(f"  ⚠️  JSON parse error: {result['parse_error']}")
    return result


def execute_sql(sql: str) -> pd.DataFrame:
    return pd.read_sql(sql, SQL_CONN)


def sql_rag(nl_query: str, table_names: List[str] = None, max_retries: int = 2):
    """
    Full SQL RAG with JSON-based generation:
      1. LLM returns JSON { sql, confident, reasoning }
      2. If confident=false or sql=null → skip execution, go straight to error rewrite
      3. If execution fails → retry with _build_retry_prompt (prev_sql + error)
      4. All retries exhausted → _build_error_rewrite_prompt (user-friendly message)
      Returns { df, sql, reasoning, markdown_table, answer, error }
    """
    sql, df, error, reasoning = None, None, None, ""

    for attempt in range(max_retries + 1):
        parsed = generate_sql_json(nl_query, table_names, error_msg=error, prev_sql=sql)

        reasoning = parsed.get("reasoning", "")
        sql       = parsed.get("sql")

        # LLM flagged it cannot answer confidently — no point executing
        if not parsed["confident"] or not sql:
            error = parsed.get("parse_error") or "LLM was not confident enough to generate a query."
            print(f"  ⚠️  Skipping execution: {error}")
            break

        print(f"  [SQL attempt {attempt+1}] {sql[:120]}")
        try:
            df    = execute_sql(sql)
            error = None
            break
        except Exception as e:
            error = str(e)
            print(f"  ❌ SQL error: {error}")

    if df is None:
        schema   = get_schema_for_llm(table_names)
        friendly = generate_with_llm(
            _build_error_rewrite_prompt(nl_query, schema, sql, error, reasoning)
        )
        return {
            "df"            : None,
            "sql"           : sql,
            "reasoning"     : reasoning,
            "markdown_table": friendly,
            "answer"        : friendly,
            "error"         : error,
        }

    md = df.to_markdown(index=False) if hasattr(df, "to_markdown") else df.to_string(index=False)
    return {"df": df, "sql": sql, "reasoning": reasoning,
            "markdown_table": md, "answer": None, "error": None}


# ── Plotly visualization ───────────────────────────────────────────────────────
def _infer_chart_type(df: pd.DataFrame, hint: str = "") -> str:
    hint_l = hint.lower()
    if any(k in hint_l for k in ["pie","share","proportion"]): return "pie"
    if any(k in hint_l for k in ["scatter","correlation","vs"]): return "scatter"
    if any(k in hint_l for k in ["line","trend","over time","time series"]): return "line"
    if any(k in hint_l for k in ["histogram","distribution","freq"]): return "histogram"
    num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    cat_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    if len(num_cols) >= 2 and len(cat_cols) == 0: return "scatter"
    if len(cat_cols) >= 1 and len(num_cols) >= 1: return "bar"
    return "bar"


def visualize_sql_result(df: pd.DataFrame, title: str = "", query_hint: str = "",
                          export_png: bool = True, export_html: bool = True) -> dict:
    if df is None or df.empty:
        print("  ⚠️  No data to visualize.")
        return {}

    chart_type  = _infer_chart_type(df, query_hint)
    num_cols    = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    cat_cols    = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    fig         = None
    chart_title = title or query_hint[:80]

    if chart_type == "bar" and cat_cols and num_cols:
        fig = px.bar(df, x=cat_cols[0], y=num_cols[0], title=chart_title,
                     color=cat_cols[0], template="plotly_dark")
    elif chart_type == "pie" and cat_cols and num_cols:
        fig = px.pie(df, names=cat_cols[0], values=num_cols[0], title=chart_title,
                     template="plotly_dark")
    elif chart_type == "scatter" and len(num_cols) >= 2:
        color_col = cat_cols[0] if cat_cols else None
        fig = px.scatter(df, x=num_cols[0], y=num_cols[1], title=chart_title,
                         color=color_col, template="plotly_dark")
    elif chart_type == "line" and num_cols:
        x_col = cat_cols[0] if cat_cols else df.columns[0]
        fig = px.line(df, x=x_col, y=num_cols[0], title=chart_title,
                      template="plotly_dark")
    elif chart_type == "histogram" and num_cols:
        fig = px.histogram(df, x=num_cols[0], title=chart_title,
                           template="plotly_dark")
    else:
        fig = px.bar(df, x=df.columns[0], y=df.columns[1] if len(df.columns) > 1 else df.columns[0],
                     title=chart_title, template="plotly_dark")

    fig.update_layout(margin=dict(l=40, r=40, t=60, b=40))
    display(fig)

    out = {"fig": fig}
    if export_html:
        html_path = f"/kaggle/working/{re.sub(chr(32),'_',chart_title[:30])}.html"
        fig.write_html(html_path)
        out["html_path"] = html_path
        print(f"  💾 HTML saved: {html_path}")
    if export_png:
        try:
            png_path = f"/kaggle/working/{re.sub(chr(32),'_',chart_title[:30])}.png"
            fig.write_image(png_path, scale=2)
            out["png_path"] = png_path
            print(f"  🖼️  PNG saved: {png_path}")
        except Exception as e:
            print(f"  ⚠️  PNG export failed (kaleido needed): {e}")
    return out


def sql_rag_with_viz(nl_query: str, table_names: List[str] = None,
                     auto_viz: bool = True) -> dict:
    """
    Combined SQL RAG + auto-visualization entry point.
    Returns {answer, df, sql, markdown_table, chart, chart_html}.
    chart_html is a self-contained Plotly HTML string for the UI chat bubble.
    """
    result = sql_rag(nl_query, table_names)
    df     = result.get("df")
    chart  = {}
    chart_html = ""

    if auto_viz and df is not None and not df.empty:
        # Notebook-side chart (for display inside Kaggle)
        chart = visualize_sql_result(df, query_hint=nl_query)

        # UI-side chart: self-contained HTML string injected into chat bubble
        try:
            import plotly.io as pio
            num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
            cat_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
            if cat_cols and num_cols and len(df) >= 2:
                chart_type = _infer_chart_type(df, nl_query)
                palette = ["#4f8ef7","#4ade80","#fbbf24","#f87171","#a78bfa",
                           "#38bdf8","#fb923c","#34d399","#e879f9","#facc15"]
                if chart_type == "bar":
                    labels = df[cat_cols[0]].astype(str).tolist()
                    values = df[num_cols[0]].tolist()
                    colors = [palette[i % len(palette)] for i in range(len(labels))]
                    fig = go.Figure(go.Bar(
                        x=labels, y=values,
                        marker_color=colors,
                        text=[f"{v:,.0f}" if isinstance(v,(int,float)) else str(v) for v in values],
                        textposition="outside",
                        hovertemplate="<b>%{x}</b><br>Value: %{y:,.0f}<extra></extra>",
                    ))
                elif chart_type == "pie":
                    labels = df[cat_cols[0]].astype(str).tolist()
                    values = df[num_cols[0]].tolist()
                    fig = go.Figure(go.Pie(labels=labels, values=values))
                elif chart_type == "line":
                    x_col = cat_cols[0] if cat_cols else df.columns[0]
                    fig = px.line(df, x=x_col, y=num_cols[0])
                else:
                    labels = df[cat_cols[0]].astype(str).tolist() if cat_cols else df.iloc[:,0].astype(str).tolist()
                    values = df[num_cols[0]].tolist() if num_cols else [0]*len(labels)
                    colors = [palette[i % len(palette)] for i in range(len(labels))]
                    fig = go.Figure(go.Bar(x=labels, y=values, marker_color=colors))

                fig.update_layout(
                    paper_bgcolor="#0f1a2e", plot_bgcolor="#0f1a2e",
                    font=dict(color="#c0d4f8", size=12),
                    title=dict(text=nl_query[:60], font=dict(size=12, color="#eef2ff"), x=0.5),
                    xaxis=dict(showgrid=False, color="#5a6a8a"),
                    yaxis=dict(showgrid=True, gridcolor="#1a2030", color="#5a6a8a", tickformat=",.0f"),
                    margin=dict(l=50, r=20, t=50, b=40),
                    height=300, bargap=0.35,
                )
                chart_html = pio.to_html(
                    fig, full_html=False, include_plotlyjs=True,
                    config={"displayModeBar": False, "responsive": True},
                )
        except Exception as e:
            print(f"  ⚠️  chart_html generation failed: {e}")

    # Convert result table → natural language answer (only on success)
    if df is not None and not df.empty:
        answer = generate_with_llm(
            _build_data_to_text_prompt(nl_query, result["markdown_table"])
        )
    else:
        answer = result.get("answer") or "No data returned."

    return {**result, "answer": answer, "chart": chart, "chart_html": chart_html}


print("✅ SQL RAG + Visualization pipeline ready")

## 🖼️ Visual RAG Pipeline (ColPali + Qwen2.5-VL-7B)

In [ ]:
from pdf2image import convert_from_path
from dataclasses import dataclass, field as dc_field
from concurrent.futures import ThreadPoolExecutor

# ── Data structures ────────────────────────────────────────────────────────────
@dataclass
class DocumentIndex:
    doc_path  : str
    pages     : list
    embeddings: object
    language  : str = "global"

@dataclass
class VisualRAGResult:
    answer       : str
    confidence   : float
    top_pages    : list
    top_scores   : list
    language_used: str
    model_used   : str = "Qwen2.5-VL-7B-Instruct"

    def __str__(self):
        pages_str  = ", ".join(f"p.{p+1}" for p in self.top_pages)
        scores_str = ", ".join(f"{s:.3f}" for s in self.top_scores)
        sep = "-"*60
        return (f"\n{'='*60}\nANSWER\n{sep}\n{self.answer}\n{sep}\n"
                f"Confidence: {self.confidence:.4f} | Pages: {pages_str}\n"
                f"Scores: {scores_str} | Lang: {self.language_used.upper()}\n{'='*60}")

# ── Language detection ────────────────────────────────────────────────────────
_INDIC_RANGES = [
    (0x0900,0x097F),(0x0980,0x09FF),(0x0A00,0x0A7F),(0x0A80,0x0AFF),
    (0x0B00,0x0B7F),(0x0B80,0x0BFF),(0x0C00,0x0C7F),(0x0C80,0x0CFF),(0x0D00,0x0D7F),
]
SUPPORTED_IMG = {".png",".jpg",".jpeg",".tiff",".tif",".bmp",".webp"}

def _count_indic(text):
    return sum(1 for ch in text if any(lo<=ord(ch)<=hi for lo,hi in _INDIC_RANGES))

def _ocr_sample(img, max_chars=600):
    try:
        import pytesseract
        return pytesseract.image_to_string(img, lang="eng+hin+ben+tam+tel",
                                           config="--oem 1 --psm 3")[:max_chars]
    except: return ""

def detect_language(pages, sample_n=5, indic_thr=0.15):
    INDIC_CODES = {"hi","bn","ta","te","mr","gu","kn","ml","pa","or","as","ne","si","ur"}
    n = len(pages)
    idxs = [int(i*n/min(sample_n,n)) for i in range(min(sample_n,n))]
    total_alpha = total_indic = 0
    ocr_worked  = False
    for idx in idxs:
        text = _ocr_sample(pages[idx])
        if text.strip():
            ocr_worked    = True
            alpha         = sum(1 for c in text if c.isalpha())
            indic         = _count_indic(text)
            total_alpha  += alpha
            total_indic  += indic
    if ocr_worked and total_alpha > 50 and total_indic/total_alpha >= indic_thr:
        return "indic"
    if not ocr_worked:
        votes = 0
        for idx in idxs:
            img  = pages[idx].convert("L")
            w,h  = img.size
            crop = img.crop((w//6,h//6,5*w//6,5*h//6))
            arr  = np.array(crop, dtype=np.float32)
            dark = (arr < 128).astype(np.float32)
            density = dark.mean()
            row_runs = []
            for row in dark:
                runs = np.diff(np.concatenate([[0],row.astype(int),[0]]))
                rl   = np.where(runs==-1)[0]-np.where(runs==1)[0]
                if len(rl): row_runs.append(rl.mean())
            run_var = float(np.var(row_runs)) if row_runs else 0.0
            if density > 0.07 and run_var > 1.4: votes += 1
        if votes/len(idxs) >= 0.4: return "indic"
    return "global"

def load_document_pages(path, dpi=150):
    p = Path(path)
    if p.is_dir():
        files = sorted(f for f in p.iterdir() if f.suffix.lower() in SUPPORTED_IMG)
        pages = [Image.open(f).convert("RGB") for f in files]
    elif p.suffix.lower() == ".pdf":
        pages = convert_from_path(str(p), dpi=dpi, fmt="RGB")
    elif p.suffix.lower() in SUPPORTED_IMG:
        pages = [Image.open(p).convert("RGB")]
    else:
        raise ValueError(f"Unsupported: {p.suffix}")
    max_side = 1600
    out = []
    for img in pages:
        w,h = img.size
        if max(w,h) > max_side:
            r   = max_side/max(w,h)
            img = img.resize((int(w*r),int(h*r)), Image.LANCZOS)
        out.append(img)
    return out

# ── ColPali embedding ─────────────────────────────────────────────────────────
@torch.no_grad()
def embed_pages_colpali(pages, batch_size=2):
    model, processor = M.colpali_model, M.colpali_processor
    n = len(pages)
    batches = [pages[i:i+batch_size] for i in range(0,n,batch_size)]
    t0 = time.perf_counter()
    def _pre(batch): return processor.process_images(batch)
    with ThreadPoolExecutor(max_workers=min(4,len(batches))) as pool:
        preprocessed = list(pool.map(_pre, batches))
    all_embs = []
    for i,(inputs_cpu,batch) in enumerate(zip(preprocessed,batches)):
        inputs = inputs_cpu.to(DEVICE)
        emb    = model(**inputs)
        all_embs.append(emb.cpu())
        del inputs, emb
        if DEVICE=="cuda": torch.cuda.empty_cache()
        pages_done = min((i+1)*batch_size, n)
        print(f"  [ColPali] Pages {i*batch_size+1}-{pages_done}/{n}")
    total_ms = (time.perf_counter()-t0)*1000
    embs = torch.cat(all_embs, dim=0)
    print(f"  [ColPali] Throughput: {n/(total_ms/1000):.1f} pages/sec")
    return embs

@torch.no_grad()
def embed_query_colpali(query):
    processor, model = M.colpali_processor, M.colpali_model
    inputs = processor.process_queries([query]).to(DEVICE)
    if "attention_mask" in inputs:
        inputs["attention_mask"] = inputs["attention_mask"].to(torch.float16)
    emb = model(**inputs)
    res = emb[0].cpu()
    del inputs, emb
    if DEVICE=="cuda": torch.cuda.empty_cache()
    return res

def maxsim_score(q_emb, p_emb):
    q   = F.normalize(q_emb.float(), dim=-1)
    p   = F.normalize(p_emb.float(), dim=-1)
    sim = torch.matmul(q, p.T)
    return sim.max(dim=1).values.sum().item()

def retrieve_top_k_visual(query, doc_index, k=COLPALI_TOP_K):
    q_emb  = embed_query_colpali(query)
    n      = doc_index.embeddings.shape[0]
    scores = [maxsim_score(q_emb, doc_index.embeddings[i]) for i in range(n)]
    ranked = sorted(enumerate(scores), key=lambda x:x[1], reverse=True)
    top    = ranked[:k]
    return [i for i,_ in top], [s for _,s in top], [doc_index.pages[i] for i,_ in top]

# ── VLM generation ────────────────────────────────────────────────────────────
VLM_SYSTEM = (
    "You are a precise document analysis assistant. "
    "Answer the user's question based ONLY on the visible content in the images. "
    "If not present, say so clearly. "
    "Preserve tables, numbers, proper names exactly. "
    "Reply in the same language as the question."
)

# ── NEW: resize helper to prevent OOM ─────────────────────────────────────────
def _resize_for_vlm(img: Image.Image, max_side: int = 896) -> Image.Image:
    w, h = img.size
    scale = min(max_side / w, max_side / h, 1.0)
    if scale < 1.0:
        img = img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)
    return img

@torch.no_grad()
def generate_visual_answer(query, page_images):
    from qwen_vl_utils import process_vision_info

    # ── OOM FIX 1: only use top-1 page for generation ─────────────────────────
    page_images = page_images[:1]

    # ── OOM FIX 2: downscale before encoding ──────────────────────────────────
    page_images = [_resize_for_vlm(img, max_side=896) for img in page_images]

    # ── OOM FIX 3: free fragmented cache before heavy forward pass ────────────
    if DEVICE == "cuda":
        gc.collect()
        torch.cuda.empty_cache()

    model, processor = M.vlm_model, M.vlm_processor
    content   = [{"type":"image","image":img} for img in page_images]
    content.append({"type":"text","text":query})
    messages  = [{"role":"system","content":VLM_SYSTEM},
                 {"role":"user","content":content}]
    text_in   = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    img_in, vid_in = process_vision_info(messages)
    inputs    = processor(text=[text_in], images=img_in, videos=vid_in,
                          padding=True, return_tensors="pt").to(model.device)
    t0 = time.perf_counter()
    out_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS,
                              do_sample=False, repetition_penalty=1.1)
    if DEVICE=="cuda": torch.cuda.synchronize()
    ms = (time.perf_counter()-t0)*1000
    n_tok = out_ids.shape[1]-inputs.input_ids.shape[1]
    print(f"  [VLM] {ms:.0f}ms | {n_tok} tokens | {n_tok/(ms/1000):.1f} tok/sec")
    trimmed = [out[len(inp):] for inp,out in zip(inputs.input_ids, out_ids)]
    return processor.batch_decode(trimmed, skip_special_tokens=True,
                                   clean_up_tokenization_spaces=False)[0].strip()

# ── Index + Query ─────────────────────────────────────────────────────────────
def index_document_visual(path, dpi=150, min_pages=3):
    print(f"\n[Visual RAG] Indexing: {Path(path).name}")
    t0    = time.perf_counter()
    pages = load_document_pages(path, dpi=dpi)
    print(f"  Pages: {len(pages)}")
    lang  = detect_language(pages)
    print(f"  Language: {lang.upper()}")
    if len(pages) < min_pages:
        dummy = torch.ones(len(pages), 1024, 128)
        return DocumentIndex(str(path), pages, dummy, lang)
    embs  = embed_pages_colpali(pages)
    mb    = embs.element_size() * embs.nelement() / 1e6
    print(f"  Shape: {tuple(embs.shape)} | Mem: {mb:.1f}MB | "
          f"Time: {(time.perf_counter()-t0)*1000:.0f}ms")
    return DocumentIndex(str(path), pages, embs, lang)

def query_document_visual(query, doc_index, top_k=COLPALI_TOP_K):
    print(f"\n[Visual RAG] Query: '{query}'")
    t0 = time.perf_counter()
    page_idx, scores, page_imgs = retrieve_top_k_visual(query, doc_index, k=top_k)
    for rank,(pi,sc) in enumerate(zip(page_idx,scores),1):
        print(f"  Rank {rank}: page {pi+1}  MaxSim={sc:.4f}")
    answer = generate_visual_answer(query, page_imgs)  # slicing to top-1 handled inside
    total  = (time.perf_counter()-t0)*1000
    result = VisualRAGResult(answer=answer, confidence=scores[0],
                              top_pages=page_idx, top_scores=scores,
                              language_used=doc_index.language)
    print(result)
    print(f"  ⏱️  Total: {total:.0f}ms")
    return result

print("✅ Visual RAG pipeline ready")

## 🔀 LangGraph Pipeline Assembly (6-way routing: GREETING | TEXT_RAG | SQL | IMAGE_RAG | FOLLOW_UP | OUT)

In [ ]:
"""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 CELL: LangGraph Pipeline  (full replacement)
 Fixes:
  1. Uses shared _store from file_store cell (no duplicate dict)
  2. MIXED source_type handled in routing hints
  3. query_document_visual extra_pdf_paths kwarg removed
  4. rag_answer extra_context kwarg removed
  5. Manual mode override respected via state["force_route"]
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
"""

from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict, Optional, List
import time, re, sys, pathlib

# ── Import from the shared file_store (same _store dict as build_ui) ──────────
import importlib
_fs = importlib.import_module("file_store")
register_files      = _fs.register_files
get_all_dfs         = _fs.get_all_dfs
get_all_pdf_texts   = _fs.get_all_pdf_texts
get_all_paths       = _fs.get_all_paths
get_combined_pdf_text = _fs.get_combined_pdf_text
list_files          = _fs.list_files

print("✅ file_store functions imported — using shared _store")


# ── State schema ──────────────────────────────────────────────────────────────
class ChatState(TypedDict):
    user_query        : str
    _thread_id        : str
    force_route       : Optional[str]   # "TEXT_RAG"|"SQL"|"IMAGE_RAG"|None
    route             : Optional[str]
    route_letter      : Optional[str]
    draft_response    : Optional[str]
    final_response    : Optional[str]
    retrieved_chunks  : Optional[list]
    rag_score         : Optional[float]
    grounded          : Optional[bool]
    sql_query         : Optional[str]
    sql_result_text   : Optional[str]
    chart_html        : Optional[str]   # FIX: carry Plotly HTML through to UI
    visual_result     : Optional[object]
    history           : Optional[List[dict]]
    source_type       : Optional[str]
    rewritten_query   : Optional[str]
    next_node         : Optional[str]
    file_names        : Optional[List[str]]


# ── Routing ───────────────────────────────────────────────────────────────────
_ROUTE_LABELS = ["A", "B", "C", "D", "E", "F"]

def _get_allowed_ids():
    return [M.gen_tokenizer.encode(l, add_special_tokens=False)[0]
            for l in _ROUTE_LABELS]

_FILE_REF_PATTERNS = re.compile(
    r"\b(summarize|summary|describe|overview|explain|analyse|analyze|"
    r"show|list|count|total|average|mean|max|min|head|tail|columns|rows|"
    r"how many|what is|what are|tell me about|give me|the file|this file|"
    r"the csv|the pdf|the excel|the data|the document|the sheet|it|this)\b",
    re.IGNORECASE,
)

def _pre_route(query: str, has_pdf: bool, has_tabular: bool) -> str | None:
    q = query.strip().lower()
    if re.fullmatch(r"(hi+|hello+|hey+|howdy|greetings|good (morning|evening|afternoon))[!.,?]*", q):
        return "A"
    if (has_pdf or has_tabular) and _FILE_REF_PATTERNS.search(query):
        if has_tabular and not has_pdf:
            return "C"
        if has_pdf and not has_tabular:
            return "B"
        tabular_words = re.compile(
            r"\b(row|column|count|total|sum|average|mean|max|min|"
            r"stat|number|numeric|value|filter|group|sort|chart|graph|"
            r"how many|sales|revenue|profit|price|quantity)\b", re.IGNORECASE
        )
        return "C" if tabular_words.search(query) else "B"
    return None

_FORCE_TO_LETTER = {"TEXT_RAG": "B", "SQL": "C", "IMAGE_RAG": "D"}

def routing_node(state: ChatState) -> ChatState:
    query       = state["user_query"]
    history     = state.get("history") or []
    source_type = state.get("source_type") or "UNKNOWN"
    thread_id   = state.get("_thread_id", "")
    force_route = state.get("force_route")   # manual override from UI buttons

    loaded_files = list_files(thread_id)
    has_pdf      = any(f.lower().endswith(".pdf") for f in loaded_files)
    has_tabular  = any(f.lower().endswith((".csv", ".xlsx", ".xls")) for f in loaded_files)

    LETTER_TO_ROUTE = {
        "A": "GREETING", "B": "TEXT_RAG", "C": "SQL",
        "D": "IMAGE_RAG", "E": "FOLLOW_UP", "F": "OUT",
    }

    # ── 1. Manual override (UI mode buttons) ──────────────────────────────────
    if force_route and force_route in _FORCE_TO_LETTER:
        letter = _FORCE_TO_LETTER[force_route]
        route  = LETTER_TO_ROUTE[letter]
        print(f"  [Router] FORCED → {letter} ({route})  [manual override]")
        return {**state, "route": route, "route_letter": letter}

    # ── 2. Pre-routing rules ──────────────────────────────────────────────────
    pre = _pre_route(query, has_pdf, has_tabular)
    if pre is not None:
        route = LETTER_TO_ROUTE[pre]
        print(f"  [Router] '{query[:60]}' → {pre} ({route})  [pre-rule]")
        return {**state, "route": route, "route_letter": pre}

    # ── 3. LLM routing ────────────────────────────────────────────────────────
    hist_txt = ""
    if history:
        hist_txt = "\n".join(
            f"User: {t['user']}\nAssistant: {t['assistant']}"
            for t in history[-3:] if t.get("user") and t.get("assistant")
        )

    if loaded_files:
        file_list = ", ".join(loaded_files)
        file_hint = (
            f"Loaded files: {file_list}\n"
            + ("Route tabular/data/spreadsheet questions to C. " if has_tabular else "")
            + ("Route document/text/PDF questions to B. " if has_pdf else "")
            + "Only use E if truly a follow-up with no file relation."
        )
    else:
        file_hint = "No files are currently loaded."

    # Handle MIXED — both SQL and TEXT/IMAGE loaded
    if source_type == "MIXED":
        source_hint = "Both TEXT and TABULAR files are loaded. Use B for text/PDF questions, C for data/spreadsheet questions."
    else:
        source_hint = {
            "TEXT_RAG" : "Active document is TEXT (PDF/DOCX). Use B.",
            "SQL"      : "Active document is TABULAR (CSV/Excel). Use C.",
            "IMAGE_RAG": "Active document is IMAGE-based. Use D.",
        }.get(source_type, "")

    prompt = f"""Classify the user query into exactly one category.

CATEGORIES:
A - Pure greeting / salutation only
B - Question about a TEXT document (prose, paragraphs, PDF content)
C - Question about TABULAR/SPREADSHEET data (numbers, rows, columns, statistics)
D - Question about scanned images or visual document content
E - Follow-up that needs prior conversation context AND is unrelated to loaded files
F - Completely out of scope

{source_hint}
{file_hint}

RULE: If files are loaded and query mentions the data/file/document → use B or C, NOT E.

{f"Recent conversation:\n{hist_txt}\n\n" if hist_txt else ""}Query: "{query}"

Answer with exactly one letter (A/B/C/D/E/F):"""

    allowed_ids = _get_allowed_ids()
    def _force(batch_id, input_ids): return allowed_ids

    inputs = M.gen_tokenizer(prompt, return_tensors="pt", add_special_tokens=True)
    inputs.pop("token_type_ids", None)
    inputs = {k: v.to(M.gen_model.device) for k, v in inputs.items()}

    with torch.no_grad():
        out = M.gen_model.generate(
            **inputs, max_new_tokens=1, do_sample=False,
            prefix_allowed_tokens_fn=_force,
            pad_token_id=M.gen_tokenizer.eos_token_id,
            return_dict_in_generate=True, output_scores=True,
        )
    letter = M.gen_tokenizer.decode(
        [out.sequences[0][inputs["input_ids"].shape[-1]]]
    ).strip()

    # Post-LLM safety overrides
    if letter == "E" and (has_pdf or has_tabular) and _FILE_REF_PATTERNS.search(query):
        letter = "C" if has_tabular else "B"
        print(f"  [Router] FOLLOW_UP overridden → {letter}")

    # source_type overrides (only when NOT mixed)
    if source_type == "SQL"         and letter in ("B", "D"): letter = "C"
    elif source_type == "IMAGE_RAG" and letter == "B":         letter = "D"
    elif source_type == "TEXT_RAG"  and letter == "D":         letter = "B"

    route = LETTER_TO_ROUTE.get(letter, "OUT")
    print(f"  [Router] '{query[:60]}' → {letter} ({route})  [LLM, source:{source_type}]")
    return {**state, "route": route, "route_letter": letter}


def select_path(state: ChatState) -> str:
    return state.get("route_letter", "F")


# ── Greeting node ─────────────────────────────────────────────────────────────
def greeting_node(state: ChatState) -> ChatState:
    reply = generate_with_llm(
        f"Reply with a short friendly greeting to: {state['user_query']}",
        is_greeting=True
    )
    sentences = re.split(r"(?<=[.!?])\s+", reply.strip())
    reply = sentences[0].strip() if sentences else reply.strip()
    if reply and reply[-1] not in ".!?":
        reply += "."
    return {**state, "draft_response": reply}


# ── Follow-up node ────────────────────────────────────────────────────────────
def followup_node(state: ChatState) -> ChatState:
    query   = state["user_query"]
    history = state.get("history") or []

    if not history:
        return {**state,
                "draft_response": "Sorry, I need more context. Please ask a complete question.",
                "next_node": "FINALIZE"}

    hist_txt = "\n".join(
        f"User: {t['user']}\nAssistant: {t['assistant']}"
        for t in history[-4:] if t.get("user")
    )
    prompt = (
        "Rewrite this follow-up as a standalone question. Do NOT answer it.\n\n"
        f"Conversation:\n{hist_txt}\n\nFollow-up: \"{query}\"\n\nRewritten question:"
    )
    rewritten = generate_with_llm(prompt, is_greeting=False).strip()
    print(f"  [FollowUp] Rewritten: '{rewritten}'")

    source_type = state.get("source_type", "TEXT_RAG")
    next_node   = {"SQL": "SQL", "IMAGE_RAG": "IMAGE_RAG", "MIXED": "SQL"}.get(source_type, "TEXT_RAG")

    return {**state, "rewritten_query": rewritten, "user_query": rewritten, "next_node": next_node}


def _followup_path(state: ChatState) -> str:
    return state.get("next_node", "TEXT_RAG")


# ── Text RAG node ─────────────────────────────────────────────────────────────
def text_rag_node(state: ChatState, rag_index: dict) -> ChatState:
    thread_id = state.get("_thread_id", "")
    query     = state["user_query"]

    uploaded_pdf_texts = get_all_pdf_texts(thread_id)

    if uploaded_pdf_texts and not rag_index:
        # No vector index — answer from raw text
        combined = get_combined_pdf_text(thread_id, max_chars_per_file=6000)
        prompt = (
            f"Answer the question using ONLY the document text below.\n\n"
            f"{combined}\n\nQuestion: {query}\n\nAnswer:"
        )
        answer = generate_with_llm(prompt, is_greeting=False)
        return {**state, "draft_response": answer, "retrieved_chunks": [],
                "rag_score": 1.0, "grounded": True}

    if rag_index:
        # FIX: rag_answer does NOT accept extra_context kwarg — removed
        result = rag_answer(query, rag_index, state.get("history"))
        # Optionally prepend raw uploaded text if answer is weak
        if uploaded_pdf_texts and not result.get("grounded"):
            combined = get_combined_pdf_text(thread_id, max_chars_per_file=2000)
            prompt = (
                f"Answer using the documents below.\n\n{combined}\n\n"
                f"Question: {query}\n\nAnswer:"
            )
            fallback = generate_with_llm(prompt, is_greeting=False)
            result["answer"] = fallback
        return {**state, "draft_response": result["answer"],
                "retrieved_chunks": result["chunks"],
                "rag_score": result["rag_score"], "grounded": result["grounded"]}

    return {**state, "draft_response": "No text document loaded. Please upload a PDF."}


# ── SQL node ──────────────────────────────────────────────────────────────────
def sql_node(state: ChatState, table_names: List[str] = None) -> ChatState:
    thread_id = state.get("_thread_id", "")
    query     = state["user_query"]

    uploaded_dfs = get_all_dfs(thread_id)

    if uploaded_dfs:
        registered_tables = []
        for fname, df in uploaded_dfs.items():
            tname = re.sub(r"\W+", "_", pathlib.Path(fname).stem.lower()).strip("_")

            # ── FIX: only load if not already present in SQL_CONN ─────────────
            cursor = SQL_CONN.cursor()
            cursor.execute(
                "SELECT name FROM sqlite_master WHERE type='table' AND name=?",
                (tname,)
            )
            already_loaded = cursor.fetchone() is not None

            if not already_loaded:
                # Sanitize column names to match what load_file_to_sql() produces
                df_clean = df.copy()
                df_clean.columns = [
                    re.sub(r"\W+", "_", c.strip().lower()) for c in df_clean.columns
                ]
                df_clean.to_sql(tname, SQL_CONN, index=False, if_exists="replace")
                print(f"  [SQL] Loaded '{tname}' → {len(df_clean)} rows × {len(df_clean.columns)} cols")
            else:
                print(f"  [SQL] '{tname}' already in DB — skipping re-insert")

            registered_tables.append(tname)

        result = sql_rag_with_viz(query, table_names=registered_tables)

    elif table_names and SQL_CONN:
        result = sql_rag_with_viz(query, table_names=table_names)
    else:
        return {**state, "draft_response": "No tabular data loaded. Please upload a CSV or Excel file."}

    df_result   = result.get("df")
    result_text = ""
    if df_result is not None:
        try:
            result_text = df_result.head(20).to_string(index=False)
        except Exception:
            result_text = str(df_result)

    return {**state,
            "draft_response" : result.get("answer", ""),
            "sql_query"      : result.get("sql", ""),
            "sql_result_text": result_text,
            "chart_html"     : result.get("chart_html", "")}  # FIX: pass chart through

# ── Image RAG node ────────────────────────────────────────────────────────────
def image_rag_node(state: ChatState, doc_index) -> ChatState:
    # FIX: query_document_visual does NOT accept extra_pdf_paths — removed
    result = query_document_visual(state["user_query"], doc_index)
    return {**state,
            "draft_response": result.answer,
            "visual_result" : result,
            "grounded"      : result.confidence > 3.0}


# ── Out node ──────────────────────────────────────────────────────────────────
def out_node(state: ChatState) -> ChatState:
    return {**state, "draft_response": (
        "Sorry, I can only answer questions about the loaded documents."
    )}


# ── Finalize node ─────────────────────────────────────────────────────────────
def finalize_node(state: ChatState) -> ChatState:
    history    = list(state.get("history") or [])
    answer     = (state.get("draft_response") or "").strip()
    chart_html = state.get("chart_html", "")   # FIX: preserve chart HTML
    if answer:
        history.append({
            "user"      : state["user_query"],
            "assistant" : answer,
            "route"     : state.get("route"),
            "timestamp" : time.time(),
        })
        history = history[-5:]
    return {**state, "history": history, "final_response": answer, "chart_html": chart_html}


# ── Graph builder ─────────────────────────────────────────────────────────────
def build_pipeline(
    rag_index   : dict      = None,
    doc_index              = None,
    table_names : List[str] = None,
    source_type : str       = None,
) -> object:
    checkpointer = InMemorySaver()
    graph        = StateGraph(ChatState)

    def _text_rag(state):
        if rag_index or get_all_pdf_texts(state.get("_thread_id", "")):
            return text_rag_node(state, rag_index)
        return {**state, "draft_response": "No text document loaded."}

    def _sql(state):
        if get_all_dfs(state.get("_thread_id", "")) or (SQL_CONN and table_names):
            return sql_node(state, table_names)
        return {**state, "draft_response": "No tabular data loaded. Please upload a CSV or Excel file."}

    def _image_rag(state):
        if doc_index:
            return image_rag_node(state, doc_index)
        return {**state, "draft_response": "No visual document loaded."}

    graph.add_node("router",    routing_node)
    graph.add_node("GREETING",  greeting_node)
    graph.add_node("TEXT_RAG",  _text_rag)
    graph.add_node("SQL",       _sql)
    graph.add_node("IMAGE_RAG", _image_rag)
    graph.add_node("FOLLOW_UP", followup_node)
    graph.add_node("OUT",       out_node)
    graph.add_node("FINALIZE",  finalize_node)

    graph.set_entry_point("router")
    graph.add_conditional_edges(
        "router", select_path,
        {"A": "GREETING", "B": "TEXT_RAG", "C": "SQL",
         "D": "IMAGE_RAG", "E": "FOLLOW_UP", "F": "OUT"},
    )
    graph.add_conditional_edges(
        "FOLLOW_UP", _followup_path,
        {"TEXT_RAG": "TEXT_RAG", "SQL": "SQL",
         "IMAGE_RAG": "IMAGE_RAG", "FINALIZE": "FINALIZE"},
    )
    for node in ["GREETING", "TEXT_RAG", "SQL", "IMAGE_RAG", "OUT"]:
        graph.add_edge(node, "FINALIZE")
    graph.add_edge("FINALIZE", END)

    app = graph.compile(checkpointer=checkpointer)
    print("✅ LangGraph pipeline compiled")
    print(f"   source_type={source_type} | rag={rag_index is not None} | "
          f"doc={doc_index is not None} | tables={table_names}")
    return app


print("✅ LangGraph nodes defined")

## 🔍 Document Ingestion Router (Smart auto-routing: Text / SQL / Visual RAG)

In [ ]:
import fitz          # PyMuPDF — pip install pymupdf
import pandas as pd
import re
from pathlib import Path

# ─────────────────────────────────────────────────────────────────────────────
#  Text-quality heuristic
#  Returns True when the extracted string is real prose/structured text
#  and False when it is blank, mojibake, or mostly non-ASCII garbage.
# ─────────────────────────────────────────────────────────────────────────────
def _is_meaningful_text(text: str, min_words: int = 30, min_ascii_ratio: float = 0.70) -> bool:
    """Decide if extracted text is usable for RAG (not garbage)."""
    if not text or not text.strip():
        return False
    # Word count check
    words = re.findall(r"\b[a-zA-Z]{2,}\b", text)
    if len(words) < min_words:
        return False
    # ASCII-ratio check — scanned / Indic PDFs that pdfminer 'extracts' are
    # often 90 %+ garbage codepoints
    printable = sum(1 for c in text if 32 <= ord(c) < 127)
    if len(text) > 0 and printable / len(text) < min_ascii_ratio:
        return False
    return True


# ─────────────────────────────────────────────────────────────────────────────
#  PDF text extractor (PyMuPDF — better than PyPDF2)
#  Ported from working-cb-accuracy-routing-in-follow-up-1.ipynb
# ─────────────────────────────────────────────────────────────────────────────
def load_pdf_pages_pymupdf(pdf_path: str) -> list:
    """
    Extract text from every page of a PDF with PyMuPDF layout preservation.
    Returns a list of {page, text} dicts (only non-empty pages).
    """
    doc   = fitz.open(pdf_path)
    pages = []
    for i, page in enumerate(doc):
        text = page.get_text("text")
        if text and text.strip():
            cleaned = "\n".join(
                line.rstrip() for line in text.splitlines() if line.strip()
            )
            pages.append({"page": i + 1, "text": cleaned})
    doc.close()
    return pages


# ─────────────────────────────────────────────────────────────────────────────
#  Unified document ingestor
# ─────────────────────────────────────────────────────────────────────────────
SUPPORTED_TABULAR   = {".xlsx", ".xls", ".xlsm", ".csv", ".tsv"}
SUPPORTED_TEXT_DOCS = {".pdf", ".docx", ".doc", ".txt", ".md"}
SUPPORTED_IMAGES    = {".png", ".jpg", ".jpeg", ".tiff", ".tif", ".bmp", ".webp"}

def ingest_document(file_path: str) -> dict:
    """
    Smart document ingestor.

    Returns
    -------
    dict with keys:
        source_type  : 'TEXT_RAG' | 'SQL' | 'IMAGE_RAG'
        rag_index    : built index (or None)
        doc_index    : DocumentIndex (or None)
        sql_tables   : list[str] of table names (or None)
        file_path    : original path
        reason       : human-readable routing decision
    """
    p    = Path(file_path)
    ext  = p.suffix.lower()
    result = dict(rag_index=None, doc_index=None, sql_tables=None,
                  file_path=str(p))

    # ── 1. Tabular files → SQL RAG ────────────────────────────────────────────
    if ext in SUPPORTED_TABULAR:
        print(f"[Ingest] Tabular file detected → SQL RAG")
        tables = load_file_to_sql(file_path)   # defined in SQL RAG cell
        result.update(source_type="SQL", sql_tables=tables,
                      reason=f"{ext} file → SQL pipeline")
        print(f"  ✅ Loaded into SQL. Tables: {tables}")
        return result

    # ── 2. Image files → Visual RAG directly ─────────────────────────────────
    if ext in SUPPORTED_IMAGES:
        print(f"[Ingest] Image file detected → Visual RAG (ColPali + VLM)")
        doc_idx = index_document_visual(file_path)  # defined in Visual RAG cell
        result.update(source_type="IMAGE_RAG", doc_index=doc_idx,
                      reason=f"{ext} image → Visual pipeline")
        return result

    # ── 3. Text documents: probe for extractable text ─────────────────────────
    if ext in SUPPORTED_TEXT_DOCS:
        extracted_text = ""
        sample_pages   = []

        if ext == ".pdf":
            try:
                sample_pages = load_pdf_pages_pymupdf(file_path)[:5]  # probe first 5 pages
                extracted_text = " ".join(p["text"] for p in sample_pages)
            except Exception as e:
                print(f"  ⚠️  PyMuPDF extraction error: {e}")

        elif ext in (".docx", ".doc"):
            try:
                import docx
                doc = docx.Document(file_path)
                extracted_text = " ".join(p.text for p in doc.paragraphs)
            except Exception as e:
                print(f"  ⚠️  DOCX extraction error: {e}")

        elif ext in (".txt", ".md"):
            extracted_text = Path(file_path).read_text(errors="ignore")

        # ── Decision ──────────────────────────────────────────────────────────
        if _is_meaningful_text(extracted_text):
            print(f"[Ingest] Good text extracted ({len(extracted_text.split())} words) → Text RAG")
            rag_idx = build_rag_index_pymupdf(file_path)  # upgraded indexer below
            result.update(source_type="TEXT_RAG", rag_index=rag_idx,
                          reason=f"{ext} with good text → Text RAG pipeline")
        else:
            print(f"[Ingest] Garbage/no text extracted → Visual RAG (ColPali + VLM)")
            doc_idx = index_document_visual(file_path)
            result.update(source_type="IMAGE_RAG", doc_index=doc_idx,
                          reason=f"{ext} with poor/no text → Visual pipeline")
        return result

    raise ValueError(f"Unsupported file type: {ext}")


print("✅ Document ingestor ready — call: result = ingest_document('/path/to/file')")

## 🧪 Unified Document Load & Pipeline Build

In [ ]:
import uuid
 
ingestion = {
    "source_type": None,
    "rag_index":   None,
    "doc_index":   None,
    "sql_tables":  None,
}
 
app = build_pipeline(
    rag_index   = None,
    doc_index   = None,
    table_names = None,
    source_type = None,
)
 
thread_id = str(uuid.uuid4())
config    = {"configurable": {"thread_id": thread_id}}
print("🎉 Pipeline ready — waiting for document upload via UI")
 
 

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 🧪 INLINE TESTER — no UI needed, run queries directly in the notebook
# Usage:
#   1. Set TEST_FILE_PATH to any local file (csv/pdf/xlsx/png …)
#   2. Run this cell → ingests the file, prints source_type
#   3. Call  quick_ask("your question")  in the next cell
# ─────────────────────────────────────────────────────────────────────────────
import uuid as _uuid

# ── ① Point to your file ─────────────────────────────────────────────────────
TEST_FILE_PATH = "/kaggle/input/datasets/dp0741470/document/RAG doc.pdf"  # ← change me

# ── ② Ingest ─────────────────────────────────────────────────────────────────
_test_thread = str(_uuid.uuid4())
_test_config = {"configurable": {"thread_id": _test_thread}}

_test_fname   = Path(TEST_FILE_PATH).name
_test_ingested = ingest_document(TEST_FILE_PATH)

# Register file in store so SQL/Text nodes can find it
import pandas as _pd_
_test_df   = _pd_.read_csv(TEST_FILE_PATH, encoding="latin-1", on_bad_lines="skip")if TEST_FILE_PATH.endswith(".csv") else None
register_files(_test_thread, {
    _test_fname: {
        "path": TEST_FILE_PATH,
        "text": _test_ingested.get("rag_index", {}) and "",
        "df"  : _test_df,
    }
})

_test_app = build_pipeline(
    rag_index   = _test_ingested.get("rag_index"),
    doc_index   = _test_ingested.get("doc_index"),
    table_names = _test_ingested.get("sql_tables"),
    source_type = _test_ingested.get("source_type"),
)
print(f"\n✅ Inline tester ready | source_type = {_test_ingested['source_type']}")
print("   Call  quick_ask('your question')  to test")

# ── ③ quick_ask helper ───────────────────────────────────────────────────────
def quick_ask(query: str, force_route: str =  'SQL'):
    """
    Run a single query through the inline-loaded pipeline and pretty-print the result.
    force_route: None | 'TEXT_RAG' | 'SQL' | 'IMAGE_RAG'
    """
    import time as _time
    t0  = _time.perf_counter()
    res = _test_app.invoke(
        {
            "user_query"  : query,
            "_thread_id"  : _test_thread,
            "file_names"  : [_test_fname],
            "force_route" : force_route,
            "chart_html"  : "",
        },
        _test_config,
    )
    ms = (_time.perf_counter() - t0) * 1000    
    route = res.get("route", "?")
    ans   = res.get("final_response", str(res))
    print(f"\n[{route}] ({ms:.0f} ms)")
    print("─" * 60)
    print(ans)
    return res


In [ ]:
print(quick_ask("who is te founder "))

In [ ]:
def _make_plotly_chart_html(labels=None, values=None, colors=None,
                             title="Query Results", chart_type="bar"):
    """
    Generate a self-contained Plotly chart HTML string (no CDN required).
    Called by the UI to render charts inline in chat bubbles.

    FIX: original version took zero params and used hardcoded demo data.
    Now accepts dynamic data from sql_rag_with_viz results.
    """
    import plotly.graph_objects as go
    import plotly.io as pio

    labels = labels or ["A", "B", "C"]
    values = values or [1, 1, 1]
    colors = colors or ["#4f8ef7", "#4ade80", "#fbbf24"]

    fig = go.Figure(go.Bar(
        x             = labels,
        y             = values,
        marker_color  = colors[:len(labels)],
        text          = [f"{v:,.0f}" if isinstance(v, (int, float)) else str(v) for v in values],
        textposition  = "outside",
        hovertemplate = "<b>%{x}</b><br>Value: %{y:,.0f}<extra></extra>",
    ))
    fig.update_layout(
        paper_bgcolor = "#0f1a2e",
        plot_bgcolor  = "#0f1a2e",
        font          = dict(color="#c0d4f8", size=12),
        title         = dict(text=title, font=dict(size=13, color="#eef2ff"), x=0.5),
        xaxis  = dict(showgrid=False, color="#5a6a8a", tickfont=dict(size=11)),
        yaxis  = dict(showgrid=True, gridcolor="#1a2030", color="#5a6a8a", tickformat=",.0f"),
        margin = dict(l=50, r=20, t=50, b=40),
        height = 300,
        bargap = 0.35,
        hoverlabel = dict(bgcolor="#1a2540", font_size=12, font_color="#eef2ff"),
    )
    return pio.to_html(
        fig,
        full_html        = False,
        include_plotlyjs = True,
        config           = {"displayModeBar": False, "responsive": True},
    )


In [ ]:
PRELOAD_CATALOG = {
    "Sample - Superstore.csv" : "/kaggle/input/datasets/dp0741470/excelsheet/Sample - Superstore.csv",
    "My PDF"                 : "/kaggle/input/datasets/dp0741470/document/RAG doc.pdf",
    "Product Images"         : "/kaggle/input/datasets/devanshipatel05/ocr-samples/PublicWaterMassMailing.pdf",
}
print(f"📂 Preload catalog: {list(PRELOAD_CATALOG.keys())}")


In [ ]:
import uuid, time, pathlib, os, sys, importlib
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

if "/kaggle/working" not in sys.path:
    sys.path.insert(0, "/kaggle/working")

_fs = importlib.import_module("file_store")
register_files        = _fs.register_files
get_all_dfs           = _fs.get_all_dfs
get_all_pdf_texts     = _fs.get_all_pdf_texts
get_combined_pdf_text = _fs.get_combined_pdf_text
list_files            = _fs.list_files


_CSS = """
<style id="idoc-css">
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600&family=JetBrains+Mono:wght@400;500&display=swap');

:root {
  --bg:        #ffffff;
  --bg-soft:   #f5f6fa;
  --bg-hover:  #eceef5;
  --bg-active: #e3e7f7;
  --border:    rgba(60,70,120,0.10);
  --border-md: rgba(60,70,120,0.18);
  --accent:    #4a6cf7;
  --accent-bg: #eef0fd;
  --accent-dk: #3254e0;
  --green:     #14a67a;
  --green-bg:  #e4f7f0;
  --amber:     #c97d08;
  --amber-bg:  #fef5e0;
  --red:       #d44;
  --purple:    #7c5cbf;
  --text-hi:   #12141c;
  --text-mid:  #4a5270;
  --text-lo:   #9099b8;
  --mono:      'JetBrains Mono', 'Consolas', monospace;
  --sans:      'Inter', system-ui, sans-serif;
  --r-sm:      5px;
  --r-md:      8px;
  --r-lg:      11px;
}

.idoc * { box-sizing: border-box; margin: 0; padding: 0; }
.idoc {
  font-family: var(--sans);
  display: flex;
  border-radius: var(--r-lg);
  overflow: hidden;
  border: 1px solid var(--border-md);
  background: var(--bg);
  box-shadow: 0 2px 12px rgba(60,80,200,.07), 0 1px 3px rgba(0,0,0,.06);
}

/* ── Sidebar ── */
.idoc-sb {
  width: 140px; min-width: 140px;
  background: var(--bg-soft);
  border-right: 1px solid var(--border);
  display: flex; flex-direction: column; overflow: hidden;
}
.idoc-logo {
  height: 38px;
  display: flex; align-items: center; gap: 7px;
  padding: 0 11px;
  border-bottom: 1px solid var(--border);
  flex-shrink: 0;
}
.idoc-logo .dot {
  width: 6px; height: 6px; border-radius: 50%;
  background: var(--green);
  box-shadow: 0 0 0 3px var(--green-bg);
}
.idoc-logo .name { font-size: 12.5px; font-weight: 600; color: var(--text-hi); letter-spacing: -.02em; }
.idoc-logo .name b { color: var(--accent); }

.idoc-sec {
  padding: 9px 10px 3px;
  font-size: 8.5px; font-weight: 700;
  color: var(--text-lo); letter-spacing: .10em; text-transform: uppercase;
  flex-shrink: 0;
}
.idoc-sess-list { flex: 1; overflow-y: auto; padding: 2px 5px 5px; }
.idoc-sess-list::-webkit-scrollbar { width: 2px; }
.idoc-sess-list::-webkit-scrollbar-thumb { background: var(--border-md); border-radius: 2px; }

.idoc-sess-item {
  padding: 5px 7px; border-radius: var(--r-md); margin-bottom: 2px; cursor: default;
  transition: background .12s;
}
.idoc-sess-item:hover { background: var(--bg-hover); }
.idoc-sess-item.active { background: var(--bg-active); }
.idoc-sess-item .sn {
  font-size: 11px; font-weight: 500; color: var(--text-mid);
  white-space: nowrap; overflow: hidden; text-overflow: ellipsis; display: block;
}
.idoc-sess-item.active .sn { color: var(--text-hi); }
.idoc-sess-item .sp {
  font-size: 9.5px; color: var(--text-lo); margin-top: 1px;
  white-space: nowrap; overflow: hidden; text-overflow: ellipsis; display: block;
}
.idoc-sb-foot {
  padding: 7px 10px; border-top: 1px solid var(--border);
  display: flex; align-items: center; gap: 6px; flex-shrink: 0;
}
.idoc-av {
  width: 20px; height: 20px; border-radius: 50%;
  background: var(--accent-bg); border: 1.5px solid rgba(74,108,247,.2);
  display: flex; align-items: center; justify-content: center;
  font-size: 7.5px; font-weight: 700; color: var(--accent); flex-shrink: 0;
}
.idoc-uname { font-size: 10.5px; color: var(--text-lo); font-weight: 500; }

/* ── Main ── */
.idoc-main { flex: 1; display: flex; flex-direction: column; min-width: 0; }

.idoc-topbar {
  height: 38px; background: var(--bg-soft);
  border-bottom: 1px solid var(--border);
  display: flex; align-items: center; padding: 0 11px; gap: 5px; flex-shrink: 0;
}
.idoc-topbar .tt  { font-size: 12px; font-weight: 600; color: var(--text-hi); }
.idoc-topbar .sep { color: var(--border-md); font-size: 13px; }
.idoc-topbar .ts  { font-size: 10px; color: var(--text-lo); font-family: var(--mono); }
.idoc-topbar .sp  { flex: 1; }

.idoc-pill {
  display: inline-flex; align-items: center; gap: 3px;
  padding: 1.5px 7px; border-radius: 99px; font-size: 9.5px;
  font-family: var(--mono); white-space: nowrap; font-weight: 500; border: 1px solid transparent;
}
.idoc-pill.file   { background: var(--accent-bg); color: var(--accent); border-color: rgba(74,108,247,.2); }
.idoc-pill.route  { background: var(--green-bg);  color: var(--green);  border-color: rgba(20,166,122,.2); }
.idoc-pill.ms     { background: var(--amber-bg);  color: var(--amber);  border-color: rgba(201,125,8,.2); }

/* ── Expand button ── */
#idoc-fs-btn {
  display: inline-flex; align-items: center; gap: 4px;
  padding: 3px 8px; border-radius: var(--r-sm);
  border: 1px solid var(--border-md);
  background: var(--bg); color: var(--text-mid);
  cursor: pointer; font-size: 10px; font-family: var(--sans); font-weight: 500;
  transition: all .15s; line-height: 1; margin-left: 4px;
}
#idoc-fs-btn:hover { background: var(--accent-bg); color: var(--accent); border-color: rgba(74,108,247,.3); }

/* ── Chat ── */
.idoc-chat {
  flex: 1; overflow-y: auto; padding: 11px 12px 6px;
  display: flex; flex-direction: column; gap: 7px;
}
.idoc-chat::-webkit-scrollbar { width: 3px; }
.idoc-chat::-webkit-scrollbar-thumb { background: var(--border); border-radius: 2px; }

.idoc-empty {
  flex: 1; display: flex; flex-direction: column;
  align-items: center; justify-content: center; gap: 8px;
}
.idoc-empty .ei {
  width: 34px; height: 34px; border-radius: var(--r-md);
  background: var(--bg-soft); border: 1px solid var(--border-md);
  display: flex; align-items: center; justify-content: center;
}
.idoc-empty p { font-size: 11.5px; color: var(--text-lo); text-align: center; max-width: 170px; line-height: 1.6; }

.idoc-msg { display: flex; gap: 7px; align-items: flex-end; }
.idoc-msg.me { flex-direction: row-reverse; }
.idoc-mav {
  width: 20px; height: 20px; min-width: 20px; border-radius: 50%;
  background: var(--bg-soft); border: 1px solid var(--border-md);
  display: flex; align-items: center; justify-content: center;
  font-size: 7.5px; font-weight: 700; color: var(--text-lo); flex-shrink: 0;
}
.idoc-bbl {
  max-width: 75%; padding: 8px 11px;
  font-size: 12px; line-height: 1.7; border-radius: var(--r-md);
}
.idoc-msg.me .idoc-bbl {
  background: linear-gradient(135deg, #eef1fd 0%, #e8ecfb 100%);
  border: 1px solid rgba(74,108,247,.18);
  border-radius: var(--r-md) var(--r-md) 3px var(--r-md);
  color: #2a3d9a;
}
.idoc-msg.ai .idoc-bbl {
  background: var(--bg-soft);
  border: 1px solid var(--border);
  border-radius: 3px var(--r-md) var(--r-md) var(--r-md);
  color: var(--text-hi);
}
.idoc-bbl .meta { display: flex; gap: 4px; flex-wrap: wrap; margin-top: 6px; }
.idoc-bbl .ftag {
  display: inline-flex; align-items: center; gap: 3px; margin-top: 4px;
  padding: 1.5px 6px; border-radius: 4px; font-size: 9.5px;
  font-family: var(--mono); background: var(--green-bg); color: var(--green);
  border: 1px solid rgba(20,166,122,.2);
}

@keyframes blink { 0%,80%,100%{opacity:.15} 40%{opacity:.9} }
.idoc-dot {
  display: inline-block; width: 4px; height: 4px; border-radius: 50%;
  background: var(--text-lo); animation: blink 1.4s infinite; margin: 0 2px;
}
.idoc-dot:nth-child(2){animation-delay:.2s}
.idoc-dot:nth-child(3){animation-delay:.4s}

/* ── Bottom bar ── */
.idoc-bottom {
  background: var(--bg-soft);
  border-top: 1px solid var(--border);
  padding: 5px 7px 7px;
}

/* Status */
.idoc-status {
  font-family: var(--mono); font-size: 10px;
  color: var(--amber); padding: 1px 6px; min-height: 14px;
}

/* ── Textarea override ── */
.idoc-bottom .widget-textarea textarea {
  background: var(--bg) !important;
  color: var(--text-hi) !important;
  border: 1.5px solid var(--border-md) !important;
  border-radius: var(--r-md) !important;
  font-family: var(--sans) !important;
  font-size: 12px !important;
  padding: 7px 10px !important;
  resize: none !important;
  outline: none !important;
  transition: border-color .15s, box-shadow .15s !important;
}
.idoc-bottom .widget-textarea textarea:focus {
  border-color: var(--accent) !important;
  box-shadow: 0 0 0 3px rgba(74,108,247,.10) !important;
}
.idoc-bottom .widget-textarea textarea::placeholder {
  color: var(--text-lo) !important; font-size: 11.5px !important;
}

/* ── Send button ── */
.idoc-send-wrap .widget-button {
  background: var(--accent) !important;
  color: #fff !important;
  border: none !important;
  border-radius: var(--r-md) !important;
  font-family: var(--sans) !important;
  font-size: 11.5px !important;
  font-weight: 600 !important;
  cursor: pointer !important;
  letter-spacing: .01em !important;
  transition: background .15s, transform .1s !important;
  box-shadow: 0 1px 4px rgba(74,108,247,.3) !important;
}
.idoc-send-wrap .widget-button:hover {
  background: var(--accent-dk) !important;
  transform: translateY(-1px) !important;
}

/* ── Generic bottom buttons (Upload, Preload) ── */
.idoc-bottom .widget-button {
  border-radius: var(--r-md) !important;
  border: 1.5px solid var(--border-md) !important;
  font-family: var(--sans) !important;
  font-size: 11px !important;
  font-weight: 500 !important;
  transition: all .13s !important;
  cursor: pointer !important;
  background: var(--bg) !important;
  color: var(--text-mid) !important;
}
.idoc-bottom .widget-button:hover {
  border-color: var(--accent) !important;
  background: var(--accent-bg) !important;
  color: var(--accent) !important;
}

/* ── Dropdown ── */
.idoc-bottom .widget-dropdown select {
  background: var(--bg) !important;
  color: var(--text-mid) !important;
  border: 1.5px solid var(--border-md) !important;
  border-radius: var(--r-md) !important;
  font-family: var(--sans) !important;
  font-size: 11px !important;
  padding: 0 7px !important;
  cursor: pointer !important;
}
.idoc-bottom .widget-dropdown select:hover { border-color: var(--accent) !important; }

/* ── Upload widget ── */
.idoc-bottom .widget-upload button {
  background: var(--bg) !important;
  color: var(--text-mid) !important;
  border: 1.5px solid var(--border-md) !important;
  border-radius: var(--r-md) !important;
  font-size: 11px !important;
  font-family: var(--sans) !important;
  font-weight: 500 !important;
  transition: all .13s !important;
}
.idoc-bottom .widget-upload button:hover {
  border-color: var(--accent) !important;
  background: var(--accent-bg) !important;
  color: var(--accent) !important;
}

/* ── Mode segment ── */
.idoc-seg {
  display: inline-flex; gap: 2px;
  background: #ecedf5;
  border-radius: var(--r-sm);
  padding: 2px;
}
.idoc-seg .widget-button {
  border-radius: 4px !important;
  border: none !important;
  font-size: 10.5px !important;
  font-weight: 500 !important;
  height: 24px !important;
  background: transparent !important;
  color: var(--text-lo) !important;
  transition: all .13s !important;
}
.idoc-seg .widget-button:hover {
  background: rgba(255,255,255,.75) !important;
  color: var(--text-mid) !important;
}

/* ── Session chips ── */
.idoc-chips .widget-button {
  border-radius: 99px !important;
  border: 1.5px solid var(--border-md) !important;
  font-size: 10.5px !important;
  font-weight: 500 !important;
  height: 22px !important;
  padding: 0 9px !important;
  background: var(--bg) !important;
  color: var(--text-mid) !important;
  transition: all .13s !important;
}
.idoc-chips .widget-button:hover {
  background: var(--bg-hover) !important;
  border-color: var(--accent) !important;
  color: var(--accent) !important;
}

/* ── New session ── */
.idoc-new .widget-button {
  border-radius: 99px !important;
  border: 1.5px dashed var(--border-md) !important;
  font-size: 10.5px !important;
  height: 22px !important;
  padding: 0 9px !important;
  background: transparent !important;
  color: var(--text-lo) !important;
  transition: all .13s !important;
}
.idoc-new .widget-button:hover {
  border-style: solid !important;
  border-color: var(--accent) !important;
  color: var(--accent) !important;
  background: var(--accent-bg) !important;
}
</style>
"""


def _e(t):
    return str(t).replace("&","&amp;").replace("<","&lt;").replace(">","&gt;").replace("\n","<br>")


def _render_chat(sessions, active_sess, loaded_files, height="420px"):
    sess_html = ""
    for sname, sdata in sessions.items():
        act  = "active" if sname == active_sess else ""
        hist = sdata.get("history", [])
        prev = next((m["text"][:26]+"…" for m in reversed(hist) if m.get("role")=="me"), "No messages yet")
        sess_html += (f'<div class="idoc-sess-item {act}">'
                      f'<span class="sn">{_e(sname)}</span>'
                      f'<span class="sp">{_e(prev)}</span>'
                      f'</div>')

    fpills = ""
    for f in loaded_files[:3]:
        stem = pathlib.Path(f).stem[:14]
        fpills += f'<span class="idoc-pill file">&#9670; {_e(stem)}</span>'

    history = sessions[active_sess].get("history", [])
    if not history:
        msgs = """<div class="idoc-empty">
          <div class="ei">
            <svg width="15" height="15" viewBox="0 0 24 24" fill="none"
              stroke="#4a6cf7" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round">
              <path d="M21 15a2 2 0 0 1-2 2H7l-4 4V5a2 2 0 0 1 2-2h14a2 2 0 0 1 2 2z"/>
            </svg>
          </div>
          <p>Upload or preload a file, then start asking questions</p>
        </div>"""
    else:
        msgs = ""
        for m in history:
            role = m.get("role", "ai")
            text = m.get("text", "")
            if role == "thinking":
                msgs += ('<div class="idoc-msg ai"><div class="idoc-mav">AI</div>'
                         '<div class="idoc-bbl">'
                         '<span class="idoc-dot"></span>'
                         '<span class="idoc-dot"></span>'
                         '<span class="idoc-dot"></span>'
                         '</div></div>')
                continue
            av    = "You" if role == "me" else "AI"
            files = m.get("files") or []
            ftags = "".join(f'<span class="ftag">&#9672; {_e(f)}</span>' for f in files)
            ms_   = m.get("ms")
            rt    = m.get("route","")
            meta  = ""
            if ms_ is not None or rt:
                meta = '<div class="meta">'
                if ms_ is not None: meta += f'<span class="idoc-pill ms">&#8987; {ms_:.0f}ms</span>'
                if rt:              meta += f'<span class="idoc-pill route">&#8594; {_e(rt)}</span>'
                meta += "</div>"
            is_html       = role == "ai" and m.get("is_html", False)
            text_rendered = text if is_html else _e(text)
            msgs += (f'<div class="idoc-msg {role}">'
                     f'<div class="idoc-mav">{av}</div>'
                     f'<div class="idoc-bbl">{text_rendered}{ftags}{meta}</div>'
                     f'</div>')

    turns = len([m for m in history if m.get("role") == "me"])

    icon_expand   = '<svg width="11" height="11" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2.2" stroke-linecap="round" stroke-linejoin="round"><polyline points="15 3 21 3 21 9"/><polyline points="9 21 3 21 3 15"/><line x1="21" y1="3" x2="14" y2="10"/><line x1="3" y1="21" x2="10" y2="14"/></svg>'
    icon_collapse = '<svg width="11" height="11" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2.2" stroke-linecap="round" stroke-linejoin="round"><polyline points="4 14 10 14 10 20"/><polyline points="20 10 14 10 14 4"/><line x1="10" y1="14" x2="3" y2="21"/><line x1="21" y1="3" x2="14" y2="10"/></svg>'

    return f"""
<div class="idoc" id="idoc-root" style="height:{height};">
  <div class="idoc-sb">
    <div class="idoc-logo">
      <div class="dot"></div>
      <div class="name">Intelli<b>Doc</b></div>
    </div>
    <div class="idoc-sec">Sessions</div>
    <div class="idoc-sess-list">{sess_html}</div>
    <div class="idoc-sb-foot">
      <div class="idoc-av">U1</div>
      <div class="idoc-uname">User 1</div>
    </div>
  </div>
  <div class="idoc-main">
    <div class="idoc-topbar">
      <span class="tt">{_e(active_sess)}</span>
      <span class="sep">·</span>
      <span class="ts">{turns} turns</span>
      <span class="sp"></span>
      {fpills}
      <button id="idoc-fs-btn" onclick="idocToggleFS()" title="Toggle fullscreen">
        <span id="idoc-fs-icon">{icon_expand}</span>
        <span id="idoc-fs-label">Expand</span>
      </button>
    </div>
    <div class="idoc-chat" id="idoc-scroll">{msgs}</div>
  </div>
</div>

<script>
(function(){{
  var e = document.getElementById('idoc-scroll');
  if (e) e.scrollTop = e.scrollHeight + 9999;

  window._idocFS = false;
  window.idocToggleFS = function() {{
    var root  = document.getElementById('idoc-root');
    var label = document.getElementById('idoc-fs-label');
    var icon  = document.getElementById('idoc-fs-icon');
    if (!root) return;
    window._idocFS = !window._idocFS;

    var svgExp = `{icon_expand}`;
    var svgCol = `{icon_collapse}`;

    if (window._idocFS) {{
      root.style.cssText = 'position:fixed;top:0;left:0;width:100vw;height:100vh;z-index:999999;border-radius:0;border:none;';
      label.textContent = 'Collapse';
      icon.innerHTML    = svgCol;
      document.getElementById('idoc-fs-btn').style.background = '#eef0fd';
      document.getElementById('idoc-fs-btn').style.color = '#4a6cf7';
    }} else {{
      root.style.cssText = 'height:{height};border-radius:11px;border:1px solid rgba(60,70,120,0.18);';
      label.textContent = 'Expand';
      icon.innerHTML    = svgExp;
      document.getElementById('idoc-fs-btn').style.background = '';
      document.getElementById('idoc-fs-btn').style.color = '';
    }}
    setTimeout(function(){{
      var s = document.getElementById('idoc-scroll');
      if (s) s.scrollTop = s.scrollHeight + 9999;
    }}, 40);
  }};
}})();
</script>
"""


def _load_file(path):
    ext = pathlib.Path(path).suffix.lower()
    if ext == ".pdf":
        try:
            import fitz
            doc = fitz.open(path)
            t   = "\n".join(p.get_text() for p in doc); doc.close(); return t, None
        except Exception: pass
        return f"[PDF: {os.path.basename(path)}]", None
    if ext in (".csv",".xlsx",".xls",".xlsm"):
        try:
            import pandas as pd
            if ext == ".csv":
                df = None
                for enc in ("utf-8","latin-1","cp1252"):
                    try: df = pd.read_csv(path, encoding=enc); break
                    except UnicodeDecodeError: continue
                if df is None: df = pd.read_csv(path, encoding="utf-8", errors="replace")
            else: df = pd.read_excel(path)
            return df.head(5).to_string(index=False), df
        except Exception as e: return f"[Error: {e}]", None
    return f"[{os.path.basename(path)}]", None


class SessionManager:
    def __init__(self):
        self._sessions = {}
        self._active   = None

    def new_session(self, name=None):
        name = name or f"Chat {len(self._sessions)+1}"
        self._sessions[name] = {"thread_id": str(uuid.uuid4()), "history": [], "files": {}}
        self._active = name
        return name

    def switch(self, name):
        if name in self._sessions: self._active = name

    def _cur(self): return self._sessions[self._active]
    def history(self): return self._cur().get("history", [])
    def thread_id(self): return self._cur()["thread_id"]
    def config(self): return {"configurable": {"thread_id": self.thread_id()}}

    def add_msg(self, role, text, meta=None):
        e = {"role": role, "text": text}
        if meta: e.update(meta)
        self._cur()["history"].append(e)

    def add_file(self, fname, path, text, df):
        self._cur()["files"][fname] = {"path": path, "text": text or "", "df": df}

    def get_files(self): return self._cur().get("files", {})


def build_ui(app_pipeline_ref, preload_session_files=None):
    SM = SessionManager()
    SM.new_session("Chat 1")

    if preload_session_files:
        for fname, rec in preload_session_files.items():
            SM.add_file(fname, rec["path"], rec.get("text",""), rec.get("df"))
        register_files(SM.thread_id(), preload_session_files)

    _state = {"mode": None, "pending": []}

    display(HTML(_CSS))

    out_chat   = widgets.Output(layout=widgets.Layout(width="100%"))
    out_status = widgets.Output(layout=widgets.Layout(min_height="14px"))

    txt = widgets.Textarea(
        placeholder="Ask a question… (Enter = send, Shift+Enter = newline)",
        layout=widgets.Layout(flex="1", height="36px", min_height="36px", max_height="90px"),
    )
    btn_send = widgets.Button(
        description="Send →",
        layout=widgets.Layout(width="68px", height="36px", flex_shrink="0"),
        style=widgets.ButtonStyle(button_color="#4a6cf7", font_color="#ffffff", font_size="11.5px"),
    )

    uploader = widgets.FileUpload(
        accept=".pdf,.csv,.xlsx,.xls,.docx,.txt,.png,.jpg,.jpeg",
        multiple=True,
        description="↑ Upload",
        layout=widgets.Layout(width="76px", height="28px", flex_shrink="0"),
    )

    _catalog = PRELOAD_CATALOG if "PRELOAD_CATALOG" in globals() else {}
    preload_dd = widgets.Dropdown(
        options=list(_catalog.keys()) or ["— no preloads —"],
        description="",
        layout=widgets.Layout(width="140px", height="28px", flex_shrink="0"),
        style={"description_width": "0px"},
    )
    btn_preload = widgets.Button(
        description="⬇ Load",
        layout=widgets.Layout(width="60px", height="28px", flex_shrink="0"),
        style=widgets.ButtonStyle(button_color="#e4f7f0", font_color="#14a67a", font_size="10.5px"),
    )

    def _on_preload(_):
        fname = preload_dd.value
        if not fname or fname == "— no preloads —": return
        fpath = _catalog.get(fname, "")
        if not fpath or not pathlib.Path(fpath).exists():
            with out_status:
                clear_output(wait=True)
                display(HTML(f'<div class="idoc-status">✗ Not found: {fpath}</div>'))
            return
        with out_status:
            clear_output(wait=True)
            display(HTML(f'<div class="idoc-status">⏳ Loading {fname}…</div>'))
        try:
            text, df = _load_file(fpath)
            SM.add_file(fname, fpath, text, df)
            if fname not in _state["pending"]: _state["pending"].append(fname)
            result = ingest_document(fpath)
            app_pipeline_ref[0] = build_pipeline(
                rag_index   = result.get("rag_index"),
                doc_index   = result.get("doc_index"),
                table_names = result.get("sql_tables"),
                source_type = result.get("source_type"),
            )
            register_files(SM.thread_id(), SM.get_files())
            print(f"[UI] ✅ Preloaded '{fname}' → {result.get('source_type')}")
        except Exception as e:
            import traceback
            print(f"[UI] ⚠️  Preload failed: {e}"); traceback.print_exc()
        with out_status: clear_output(wait=True)
        _render()
    btn_preload.on_click(_on_preload)

    # Mode buttons — compact for small screens
    _mode_defs = [
        ("Auto",   None,         "#eef0fd", "#4a6cf7"),
        ("Text",   "TEXT_RAG",   "#e4f0fc", "#1a7abf"),
        ("Data",   "SQL",        "#fef5e0", "#c97d08"),
        ("Visual", "IMAGE_RAG",  "#f0ecfe", "#7c5cbf"),
    ]
    mode_btns = []
    for label, mval, bg_act, fg in _mode_defs:
        b = widgets.Button(
            description=label,
            layout=widgets.Layout(height="24px", width="50px", flex_shrink="0"),
            style=widgets.ButtonStyle(
                button_color=bg_act if _state["mode"] == mval else "transparent",
                font_color=fg if _state["mode"] == mval else "#9099b8",
                font_size="10.5px"),
        )
        mode_btns.append((b, mval, bg_act, fg))

    sess_btns_box = widgets.HBox(layout=widgets.Layout(flex_wrap="wrap", gap="3px"))
    btn_new_sess  = widgets.Button(
        description="＋ New",
        layout=widgets.Layout(height="22px", width="50px"),
        style=widgets.ButtonStyle(button_color="transparent", font_color="#9099b8", font_size="10px"),
    )

    def _render():
        loaded = list_files(SM.thread_id())
        with out_chat:
            clear_output(wait=True)
            display(HTML(_render_chat(SM._sessions, SM._active, loaded, height="380px")))

    def _rebuild_sess_buttons():
        btns = []
        for sname in SM._sessions:
            is_act = (sname == SM._active)
            b = widgets.Button(
                description=sname,
                layout=widgets.Layout(height="22px", max_width="80px"),
                style=widgets.ButtonStyle(
                    button_color="#e3e7f7" if is_act else "#ffffff",
                    font_color="#4a6cf7"   if is_act else "#9099b8",
                    font_size="10.5px"),
            )
            def _cb(n):
                def fn(_):
                    SM.switch(n)
                    register_files(SM.thread_id(), SM.get_files())
                    _rebuild_sess_buttons(); _render()
                return fn
            b.on_click(_cb(sname))
            btns.append(b)
        sess_btns_box.children = btns + [btn_new_sess]

    def _update_mode_styles():
        for b, mval, bg_act, fg in mode_btns:
            is_act = (_state["mode"] == mval)
            b.style.button_color = bg_act if is_act else "transparent"
            b.style.font_color   = fg     if is_act else "#9099b8"

    for b, mval, bg_act, fg in _mode_defs:
        def _mcb(mv):
            def fn(_):
                _state["mode"] = mv
                _update_mode_styles()
                lbl = {None:"Auto","TEXT_RAG":"Text RAG","SQL":"SQL · Data","IMAGE_RAG":"Visual RAG"}.get(mv,"Auto")
                with out_status:
                    clear_output(wait=True)
                    display(HTML(f'<div class="idoc-status">Mode → {lbl}</div>'))
            return fn
        for b2, mv2, *_ in mode_btns:
            if mv2 == mval: b2.on_click(_mcb(mval)); break

    def _on_new_sess(_):
        name = f"Chat {len(SM._sessions)+1}"
        SM.new_session(name)
        if preload_session_files:
            for fname, rec in preload_session_files.items():
                SM.add_file(fname, rec["path"], rec.get("text",""), rec.get("df"))
            register_files(SM.thread_id(), preload_session_files)
        _rebuild_sess_buttons(); _render()
    btn_new_sess.on_click(_on_new_sess)

    def _on_upload(change):
        uploaded = uploader.value
        pairs = (uploaded.items() if isinstance(uploaded, dict)
                 else [(u["name"], u) for u in uploaded])
        for fname, fdata in pairs:
            content = fdata.get("content", b"")
            path    = f"/tmp/idoc_{uuid.uuid4().hex[:6]}_{fname}"
            with open(path, "wb") as f: f.write(content)
            text, df = _load_file(path)
            SM.add_file(fname, path, text, df)
            if fname not in _state["pending"]: _state["pending"].append(fname)
            with out_status:
                clear_output(wait=True)
                display(HTML(f'<div class="idoc-status">⏳ Ingesting {fname}…</div>'))
            try:
                result = ingest_document(path)
                app_pipeline_ref[0] = build_pipeline(
                    rag_index   = result.get("rag_index"),
                    doc_index   = result.get("doc_index"),
                    table_names = result.get("sql_tables"),
                    source_type = result.get("source_type"),
                )
                print(f"[UI] ✅ Ingested '{fname}' → {result.get('source_type')}")
            except Exception as e:
                import traceback
                print(f"[UI] ⚠️  Ingestion failed for '{fname}': {e}"); traceback.print_exc()
        register_files(SM.thread_id(), SM.get_files())
        with out_status: clear_output(wait=True)
        _render()
    uploader.observe(_on_upload, names="value")

    def _do_send(_=None):
        q = txt.value.strip()
        if not q: return
        txt.value = ""

        attached      = list(_state["pending"]); _state["pending"] = []
        session_files = SM.get_files()
        thread_id     = SM.thread_id()
        register_files(thread_id, session_files)
        file_names    = list(session_files.keys())

        SM.add_msg("me", q, {"files": attached} if attached else None)
        SM.add_msg("thinking", "")
        _render()

        t0 = time.perf_counter()
        ans = route = ""; is_html = False

        try:
            force_map = {None: None, "TEXT_RAG": "TEXT_RAG",
                         "SQL": "SQL", "IMAGE_RAG": "IMAGE_RAG"}
            res = app_pipeline_ref[0].invoke(
                {"user_query": q, "_thread_id": thread_id,
                 "file_names": file_names,
                 "force_route": force_map.get(_state["mode"]),
                 "chart_html": ""},
                SM.config()
            )
            ans        = res.get("final_response", str(res))
            route      = res.get("route", "")
            chart_html = res.get("chart_html", "")
            if chart_html:
                ans = ans + "<br>" + chart_html; is_html = True
        except Exception as e:
            import traceback
            ans = f"⚠ {type(e).__name__}: {e}"; route = "error"
            print(f"[UI] Pipeline error:\n{traceback.format_exc()}")

        ms   = (time.perf_counter() - t0) * 1000
        hist = SM._cur()["history"]
        hist[:] = [m for m in hist if m.get("role") != "thinking"]
        SM.add_msg("ai", ans, {"ms": ms, "route": route, "is_html": is_html})
        _rebuild_sess_buttons(); _render()

    btn_send.on_click(_do_send)

    def _on_key(change):
        val = txt.value
        if val.endswith("\n") and not val.endswith("\n\n"):
            txt.value = val[:-1]; _do_send()
    txt.observe(_on_key, names="value")

    _rebuild_sess_buttons()
    _render()

    bottom_wrapper = widgets.Output(
        layout=widgets.Layout(width="100%", background_color="#f5f6fa")
    )

    # Session row
    chips_wrap = widgets.HTML('<div class="idoc-chips" style="display:inline-flex;flex-wrap:wrap;gap:3px;align-items:center;">')
    chips_end  = widgets.HTML('</div>')
    new_wrap   = widgets.HTML('<div class="idoc-new" style="display:inline-flex;">')
    new_end    = widgets.HTML('</div>')

    # Mode segment
    seg_wrap = widgets.HTML('<div class="idoc-seg" style="display:inline-flex;">')
    seg_end  = widgets.HTML('</div>')

    preload_row = widgets.HBox(
        [preload_dd, btn_preload, uploader],
        layout=widgets.Layout(gap="4px", align_items="center", padding="1px 7px 2px"),
    )

    send_wrap = widgets.HTML('<div class="idoc-send-wrap" style="display:contents;">')
    send_end  = widgets.HTML('</div>')

    input_row = widgets.HBox(
        [send_wrap, txt, btn_send, send_end],
        layout=widgets.Layout(display="flex", align_items="flex-end",
                              gap="6px", padding="0 7px 7px"),
    )

    with bottom_wrapper:
        display(HTML('<div class="idoc-bottom">'))
        display(widgets.VBox(
            [
                widgets.HBox(
                    [chips_wrap, sess_btns_box, new_wrap, btn_new_sess, new_end, chips_end],
                    layout=widgets.Layout(align_items="center", padding="3px 7px 2px")
                ),
                widgets.HBox(
                    [seg_wrap,
                     widgets.HBox([b for b, *_ in mode_btns], layout=widgets.Layout(gap="2px")),
                     seg_end],
                    layout=widgets.Layout(align_items="center", padding="0 7px 2px")
                ),
                preload_row,
                out_status,
                input_row,
            ],
            layout=widgets.Layout(background_color="transparent", padding="0"),
        ))
        display(HTML('</div>'))

    return widgets.VBox(
        [out_chat, bottom_wrapper],
        layout=widgets.Layout(
            width="100%",
            border="1px solid rgba(60,70,120,0.15)",
            border_radius="11px",
            overflow="hidden",
            background_color="#f5f6fa",
        ),
    )


# ── Launch ──────────────────────────────────────────────────────────────────
app_ref = [app]
root    = build_ui(app_ref)
# display(root)
# print("✅ IntelliDoc UI launched")

In [ ]:
"""
IntelliDoc AI — Gradio UI v14  (Gradio 3.50.2)
===============================================
FIXES v14:
1. NoneType strip on upload: all dict values from _ingest/_build coerced to
   str/empty. source_type default "TEXT_RAG" if None.
2. Dropdown not selecting: instead of relying on JS 'change' event (lost on
   Gradio re-render), the <select> posts its value directly via a Gradio
   gr.Dropdown (visible=False) synced by JS. Actually simpler fix: we render
   the select with an onchange inline handler that writes to the hidden input
   every time, AND we store the last selected value in window.__lastPreload
   so the Load button always reads it.
3. Dropdown text dark: explicit color:#1c1a17 on <select> and <option>.
"""

import os, uuid, time, pathlib, base64
import gradio as gr

try:
    _ingest = ingest_document   # noqa: F821
    _build  = build_pipeline    # noqa: F821
except NameError:
    def _ingest(path):
        return {"rag_index": None, "doc_index": None,
                "sql_tables": [], "source_type": "TEXT_RAG"}
    def _build(rag_index=None, doc_index=None, table_names=None, source_type=None):
        class _D:
            def invoke(self, inp, config=None):
                q = inp.get("user_query", "")
                return {"final_response": f"[Demo] You asked: {q}",
                        "route": source_type or "TEXT_RAG", "chart_html": ""}
        return _D()

PRELOAD_CATALOG: dict = {
    "digital PDF.pdf": "/kaggle/input/datasets/dp0741470/document/RAG doc.pdf",
    "OCR image.jpg":   "/kaggle/input/datasets/devanshipatel05/ocr-samples/0000114117.jpg",
    "CSV sheet":       "/kaggle/input/datasets/dp0741470/excelsheet/Sample - Superstore.csv",
}

_MODE_MAP = {
    "Auto":       None,
    "Text RAG":   "TEXT_RAG",
    "SQL / Data": "SQL",
    "Visual RAG": "IMAGE_RAG",
}

# ── CSS ────────────────────────────────────────────────────────────────────
_CSS = """
@import url('https://fonts.googleapis.com/css2?family=Geist+Mono:wght@400;500&family=DM+Sans:opsz,wght@9..40,300;9..40,400;9..40,500;9..40,600&display=swap');
*,*::before,*::after{box-sizing:border-box;margin:0;padding:0}
:root{
  --bg:#f4f2ee;--bg2:#ffffff;--bg3:#edeae4;--bg4:#e5e1d8;
  --bd:#ddd9d0;--bd2:#c8c3b8;
  --tx:#1c1a17;--tx2:#5c5750;--tx3:#9c9890;
  --acc:#7c5c2e;--acc2:#a07840;--accbg:#f5ede0;
  --grn:#1a7a4a;--grnbg:#e6f5ee;--gnbd:#b8dfc8;
  --blu:#2d5fa8;--blubg:#e6eef8;--blubd:#b8ccee;
  --r:8px;--f:'DM Sans',system-ui,sans-serif;--m:'Geist Mono',monospace;
  --sh:0 1px 3px rgba(0,0,0,0.07);--sh2:0 2px 8px rgba(0,0,0,0.12)
}
html,body{background:var(--bg)!important}
.gradio-container{width:100%!important;max-width:100%!important;padding:0!important;margin:0!important;background:var(--bg)!important;font-family:var(--f)!important}
.gradio-container>.contain,.gradio-container>.contain>div,.gradio-container>.contain>div>div{padding:0!important;gap:0!important;background:var(--bg)!important}
#component-0{padding:0!important;gap:0!important;background:transparent!important;border:none!important;box-shadow:none!important}
.gr-block,.gr-box,.gr-form{border:none!important;background:transparent!important;box-shadow:none!important;padding:0!important;margin:0!important}
footer{display:none!important}.gr-prose{display:none!important}

#idoc-page{display:flex;flex-direction:column;width:100%;background:var(--bg);font-family:var(--f);color:var(--tx)}
#idoc-hdr{height:46px;flex-shrink:0;background:var(--bg2);border-bottom:1px solid var(--bd);padding:0 18px;display:flex;align-items:center;gap:12px;z-index:20;box-shadow:var(--sh)}
.logo{display:flex;align-items:center;gap:8px}
.logo-mark{width:20px;height:20px;border-radius:5px;border:1.5px solid var(--acc2);background:var(--accbg);display:flex;align-items:center;justify-content:center}
.logo-mark::after{content:'';width:6px;height:6px;background:var(--acc);border-radius:1px;transform:rotate(45deg);display:block}
.logo-tx{font-size:14px;font-weight:600;color:var(--tx);letter-spacing:-.04em}
.logo-tx em{font-style:normal;color:var(--acc)}
.hsp{flex:1}
.ver{font-family:var(--m);font-size:9px;color:var(--tx3);padding:2px 7px;border:1px solid var(--bd);border-radius:4px;background:var(--bg3)}
.sdot{width:6px;height:6px;border-radius:50%;background:var(--grn);box-shadow:0 0 0 2px rgba(26,122,74,.15)}
#idoc-body{display:flex;flex:1;overflow:hidden;min-height:0}
#idoc-sb{width:196px;flex-shrink:0;background:var(--bg2);border-right:1px solid var(--bd);display:flex;flex-direction:column;overflow:hidden}
.sb-sec{flex-shrink:0;padding:10px 12px 5px;font-size:8px;font-weight:600;color:var(--tx3);letter-spacing:.14em;text-transform:uppercase;font-family:var(--m)}
.sess-scroll{overflow-y:auto;padding:0 8px 6px;max-height:200px;min-height:60px;border-bottom:1px solid var(--bd);flex-shrink:0}
.sess-scroll::-webkit-scrollbar{width:3px}
.sess-scroll::-webkit-scrollbar-thumb{background:var(--bd2);border-radius:3px}
.sess-chip{padding:7px 9px;border-radius:var(--r);margin-bottom:3px;background:var(--bg3);border:1px solid var(--bd);cursor:pointer;transition:all .12s;display:flex;flex-direction:column;gap:2px;box-shadow:0 1px 3px rgba(0,0,0,.06)}
.sess-chip:hover{border-color:var(--bd2);background:var(--bg4);box-shadow:var(--sh2)}
.sess-chip.active{background:var(--accbg);border-color:var(--acc2)}
.sc-name{font-size:11px;font-weight:500;color:var(--tx);white-space:nowrap;overflow:hidden;text-overflow:ellipsis}
.sess-chip.active .sc-name{color:var(--acc)}
.sc-meta{display:flex;gap:5px;align-items:center;font-size:8.5px;color:var(--tx3);font-family:var(--m)}
.sc-tag{background:var(--bg4);border:1px solid var(--bd);border-radius:3px;padding:0 4px;font-size:7.5px}
.file-scroll{flex:1;overflow-y:auto;padding:0 8px 8px}
.file-chip{padding:7px 9px;border-radius:var(--r);margin-bottom:3px;background:var(--bg3);border:1px solid var(--bd);display:flex;flex-direction:column;gap:3px}
.fc-row{display:flex;align-items:center;gap:6px}
.fc-icon{width:14px;height:14px;border-radius:3px;background:var(--accbg);border:1px solid var(--bd2);display:flex;align-items:center;justify-content:center;font-size:7.5px;color:var(--acc);font-family:var(--m);font-weight:600;flex-shrink:0}
.fc-name{font-size:10.5px;font-weight:500;color:var(--tx2);white-space:nowrap;overflow:hidden;text-overflow:ellipsis}
.fc-type{font-size:8px;color:var(--tx3);font-family:var(--m);padding-left:20px}
.no-items{font-size:11px;color:var(--tx3);padding:10px 6px;text-align:center;line-height:1.7;font-style:italic}
.sb-foot{flex-shrink:0;padding:8px 12px;border-top:1px solid var(--bd);display:flex;align-items:center;gap:8px}
.uav{width:20px;height:20px;border-radius:50%;background:var(--accbg);border:1px solid var(--bd2);display:flex;align-items:center;justify-content:center;font-size:7.5px;font-weight:700;color:var(--acc);font-family:var(--m)}
.ulbl{font-size:10px;color:var(--tx3)}
#idoc-main{flex:1;display:flex;flex-direction:column;overflow:hidden;min-width:0}
#idoc-topbar{height:38px;flex-shrink:0;background:var(--bg2);border-bottom:1px solid var(--bd);padding:0 16px;display:flex;align-items:center;gap:9px}
.sname{font-size:12px;font-weight:500;color:var(--tx);letter-spacing:-.02em}
.ssep{color:var(--tx3);font-size:10px}
.sturns{font-size:9.5px;color:var(--tx3);font-family:var(--m)}
.tpill{display:inline-flex;align-items:center;gap:3px;padding:2px 7px;border-radius:4px;font-size:9px;font-family:var(--m);border:1px solid transparent}
.tpf{background:var(--accbg);color:var(--acc);border-color:var(--bd2)}
.tpr{background:var(--grnbg);color:var(--grn);border-color:var(--gnbd)}
.tpms{background:var(--blubg);color:var(--blu);border-color:var(--blubd)}
#idoc-chat{overflow-y:scroll;overflow-x:hidden;padding:18px 20px;display:flex;flex-direction:column;gap:13px;scroll-behavior:smooth;background:var(--bg);min-height:300px;flex:1}
#idoc-chat::-webkit-scrollbar{width:5px}
#idoc-chat::-webkit-scrollbar-thumb{background:var(--bd2);border-radius:3px}
.empty-wrap{flex:1;display:flex;flex-direction:column;align-items:center;justify-content:center;gap:12px;text-align:center;padding:60px 20px}
.empty-glyph{width:40px;height:40px;border-radius:var(--r);border:1px solid var(--bd2);background:var(--bg2);display:flex;align-items:center;justify-content:center;color:var(--tx3);font-size:18px;box-shadow:var(--sh)}
.empty-t{font-size:12.5px;color:var(--tx2);line-height:1.65}
.empty-h{font-size:9.5px;color:var(--tx3);font-family:var(--m);letter-spacing:.04em}
.msg{display:flex;gap:9px;align-items:flex-start;animation:mi .16s ease}
@keyframes mi{from{opacity:0;transform:translateY(4px)}to{opacity:1;transform:none}}
.msg.me{flex-direction:row-reverse}
.mav{width:22px;height:22px;min-width:22px;border-radius:5px;display:flex;align-items:center;justify-content:center;font-size:7px;font-weight:700;font-family:var(--m);flex-shrink:0;margin-top:2px}
.mav.ai{background:var(--bg3);border:1px solid var(--bd2);color:var(--tx2)}
.mav.me{background:var(--accbg);border:1px solid var(--bd2);color:var(--acc)}
.bbl{max-width:74%;padding:10px 13px;font-size:12.5px;line-height:1.75;border-radius:var(--r)}
.msg.me .bbl{background:var(--bg2);border:1px solid var(--bd);border-radius:var(--r) var(--r) 3px var(--r);color:var(--tx);box-shadow:var(--sh)}
.msg.ai .bbl{background:var(--accbg);border:1px solid var(--bd);border-radius:3px var(--r) var(--r) var(--r);color:var(--tx);box-shadow:var(--sh);max-width:82%}
.meta{display:flex;gap:5px;flex-wrap:wrap;margin-top:8px;padding-top:8px;border-top:1px solid var(--bd)}
.ftag{display:inline-flex;align-items:center;gap:3px;padding:2px 6px;border-radius:4px;font-size:9px;font-family:var(--m);background:var(--grnbg);color:var(--grn);border:1px solid var(--gnbd)}
.chart-frame{width:100%;min-height:360px;border:none;border-radius:6px;margin-top:10px;display:block;background:#fff}
#idoc-bottom{flex-shrink:0;background:var(--bg2)!important;border-top:1px solid var(--bd);padding:8px 14px 10px;display:flex;flex-direction:column;gap:5px;z-index:10}
#idoc-status{font-family:var(--m);font-size:10px;color:var(--tx2);line-height:1.4;min-height:16px}
#idoc-status>div,#idoc-status p{margin:0!important;padding:0!important;background:transparent!important;color:var(--tx2)!important;font-size:10px!important}
#idoc-chips{display:flex!important;align-items:center!important;gap:4px!important;flex-shrink:0;height:28px;background:transparent!important}
#idoc-chips,#idoc-chips>*,#idoc-chips .gr-form,#idoc-chips .gr-box,#idoc-chips fieldset,#idoc-chips .block,#idoc-chips .gr-block,#idoc-chips>div,#idoc-chips>div>div{background:transparent!important;border:none!important;box-shadow:none!important;padding:0!important;margin:0!important;height:auto!important}
#idoc-chips fieldset>div,#idoc-chips .gr-radio-row{display:flex!important;gap:4px!important;flex-wrap:nowrap!important;align-items:center!important;background:transparent!important}
#idoc-chips label{display:inline-flex!important;align-items:center!important;justify-content:center!important;padding:0 10px!important;height:26px!important;border-radius:5px!important;border:1px solid var(--bd)!important;background:var(--bg3)!important;color:var(--tx)!important;font-size:10.5px!important;font-weight:500!important;cursor:pointer!important;font-family:var(--f)!important;transition:all .1s!important;white-space:nowrap!important;line-height:1!important;user-select:none!important;box-shadow:0 1px 3px rgba(0,0,0,.08)!important}
#idoc-chips label>span{display:inline!important;font-size:10.5px!important;color:inherit!important;pointer-events:none!important}
#idoc-chips label:has(input:checked){background:var(--accbg)!important;color:var(--acc)!important;border-color:var(--acc2)!important;font-weight:600!important}
#idoc-chips label:hover:not(:has(input:checked)){background:var(--bg4)!important;color:var(--tx)!important;border-color:var(--bd2)!important}
#idoc-chips label input[type=radio]{display:none!important}
#idoc-chips .label-wrap,#idoc-chips>.block>span{display:none!important}
#idoc-actions{display:flex!important;align-items:center!important;gap:5px!important;flex-shrink:0;height:32px;background:transparent!important}
#idoc-actions,#idoc-actions>*,#idoc-actions .gr-form,#idoc-actions .gr-box,#idoc-actions .block,#idoc-actions .gr-block{background:transparent!important;border:none!important;box-shadow:none!important;padding:0!important;margin:0!important}
#idoc-actions .gr-block>label,#idoc-actions .block>label,#idoc-actions .label-wrap{display:none!important}
#idoc-sel-wrap,#idoc-sel-wrap>*,#idoc-sel-wrap .gr-block,#idoc-sel-wrap .block{background:transparent!important;border:none!important;box-shadow:none!important;padding:0!important;margin:0!important;flex:1!important;min-width:0!important;display:flex!important;align-items:center!important}

/* SELECT — dark text forced, works on all OS */
#preload-native-sel{
  flex:1;width:100%;height:30px;
  background:#edeae4;
  color:#1c1a17 !important;
  border:1.5px solid #c8c3b8;border-radius:5px;
  font-family:'DM Sans',system-ui,sans-serif;
  font-size:11px;font-weight:500;
  padding:0 8px;cursor:pointer;outline:none;
  box-shadow:0 1px 3px rgba(0,0,0,.08);
  -webkit-appearance:auto;appearance:auto;
  /* Force dark text on webkit (macOS/iOS renders grey otherwise) */
  -webkit-text-fill-color:#1c1a17;
}
#preload-native-sel:hover{border-color:#a07840}
#preload-native-sel option{
  color:#1c1a17 !important;
  background:#ffffff;
  font-weight:500;
}

#idoc-actions button,#idoc-actions .upload-btn button,#idoc-actions .gr-block button,#idoc-actions .block button{height:30px!important;padding:0 12px!important;background:var(--bg3)!important;color:var(--tx)!important;border:1.5px solid var(--bd2)!important;border-radius:5px!important;font-family:var(--f)!important;font-size:11px!important;font-weight:500!important;white-space:nowrap!important;cursor:pointer!important;transition:all .1s!important;flex-shrink:0!important;box-shadow:0 1px 4px rgba(0,0,0,.10)!important}
#idoc-actions button:hover,#idoc-actions .upload-btn button:hover{background:var(--bg4)!important;color:var(--acc)!important;border-color:var(--acc2)!important}
#idoc-inp{display:flex!important;gap:7px!important;align-items:flex-end!important;flex-shrink:0;background:transparent!important}
#idoc-inp,#idoc-inp>*,#idoc-inp .gr-block,#idoc-inp .gr-box,#idoc-inp .gr-text-input,#idoc-inp .block{background:transparent!important;border:none!important;box-shadow:none!important;padding:0!important;margin:0!important}
#idoc-inp>.gr-block,#idoc-inp>.block{flex:1!important;min-width:0!important}
#idoc-inp .label-wrap{display:none!important}
#idoc-inp textarea,#idoc-inp input[type=text]{width:100%!important;display:block!important;background:var(--bg3)!important;color:var(--tx)!important;border:1.5px solid var(--bd)!important;border-radius:7px!important;font-family:var(--f)!important;font-size:13px!important;padding:8px 12px!important;resize:none!important;outline:none!important;min-height:36px!important;max-height:64px!important;overflow-y:auto!important;line-height:1.55!important;caret-color:var(--acc)!important;transition:border-color .12s,box-shadow .12s!important}
#idoc-inp textarea:focus,#idoc-inp input[type=text]:focus{border-color:var(--acc2)!important;box-shadow:0 0 0 3px rgba(160,120,64,.1)!important;background:var(--bg2)!important}
#idoc-inp textarea::placeholder{color:var(--tx3)!important;font-size:12px!important}
#send-btn{flex-shrink:0!important}
#send-btn button{background:var(--acc)!important;color:#fff!important;border:none!important;border-radius:7px!important;font-family:var(--f)!important;font-size:12px!important;font-weight:600!important;height:36px!important;padding:0 16px!important;white-space:nowrap!important;cursor:pointer!important;transition:all .1s!important;box-shadow:0 2px 6px rgba(124,92,46,.25)!important}
#send-btn button:hover{background:#6a4e26!important;transform:translateY(-1px)!important}
#idoc-bottom .block,#idoc-bottom .gr-block,#idoc-bottom .gr-box,#idoc-bottom .gr-form{background:transparent!important;border:none!important;box-shadow:none!important;padding:0!important;margin:0!important}
"""

# ── JS (injected once into chat HTML) ──────────────────────────────────────
_RESIZE_JS = """
<script>
(function(){
  /* Layout sizing */
  function layout(){
    var page=document.getElementById('idoc-page');
    var chat=document.getElementById('idoc-chat');
    var hdr=document.getElementById('idoc-hdr');
    var topbar=document.getElementById('idoc-topbar');
    var bottom=document.getElementById('idoc-bottom');
    if(!page||!chat||!hdr||!topbar||!bottom)return;
    var c=page.parentElement;
    while(c&&c.offsetHeight<200)c=c.parentElement;
    var total=c?c.offsetHeight:window.innerHeight;
    page.style.height=total+'px';
    document.getElementById('idoc-body').style.height=(total-hdr.offsetHeight)+'px';
    chat.style.height=Math.max(total-hdr.offsetHeight-topbar.offsetHeight-bottom.offsetHeight-2,300)+'px';
    chat.scrollTop=chat.scrollHeight+9999;
  }
  layout();
  window.addEventListener('resize',layout);
  if(window.ResizeObserver)new ResizeObserver(layout).observe(document.querySelector('.gradio-container')||document.body);
  [100,500,1500].forEach(function(t){setTimeout(layout,t)});

  /* ── helper: reliably set a Gradio 3.x hidden textbox value ── */
  function setGradioText(selector, val){
    var el=document.querySelector(selector);
    if(!el)return false;
    var s=Object.getOwnPropertyDescriptor(window.HTMLTextAreaElement.prototype,'value')||
           Object.getOwnPropertyDescriptor(window.HTMLInputElement.prototype,'value');
    if(s&&s.set)s.set.call(el,val);else el.value=val;
    el.dispatchEvent(new Event('input',{bubbles:true}));
    el.dispatchEvent(new Event('change',{bubbles:true}));
    return true;
  }

  /* ── Session switch ── */
  window.__pendingSid=null;
  setInterval(function(){
    if(!window.__pendingSid)return;
    var sid=window.__pendingSid;window.__pendingSid=null;
    setGradioText('#sw-inp textarea,#sw-inp input',sid);
  },120);

  /* ── Preload select ──
     Keep last chosen value in window.__lastPreload.
     On change: also immediately push to hidden textbox.
     On Load button click: push __lastPreload to textbox then click the
     hidden gr.Button — we hook the visible Load button via delegation.
  */
  window.__lastPreload='';

  /* Re-attach select handler after every Gradio re-render */
  function attachSelect(){
    var sel=document.getElementById('preload-native-sel');
    if(!sel||sel._idoc_attached)return;
    sel._idoc_attached=true;
    sel.addEventListener('change',function(){
      window.__lastPreload=sel.value;
      /* push immediately so change-triggered callbacks also work */
      setGradioText('#preload-inp textarea,#preload-inp input', sel.value);
    });
  }
  /* poll for select element (Gradio re-renders DOM) */
  setInterval(attachSelect, 300);
  attachSelect();

  /* Hook the Load button: push current __lastPreload before submitting */
  document.addEventListener('click',function(e){
    /* find if click is on or inside the Load button rendered by Gradio */
    var btn=e.target.closest('button');
    if(!btn)return;
    if(btn.textContent.trim().startsWith('↓ Load')){
      if(window.__lastPreload){
        /* write value, then let Gradio's own click handler fire normally */
        setGradioText('#preload-inp textarea,#preload-inp input', window.__lastPreload);
      }
    }
  }, true); /* capture phase so we fire before Gradio's handler */

})();
</script>
"""

# ── helpers ────────────────────────────────────────────────────────────────
def _e(t):
    return str(t).replace("&","&amp;").replace("<","&lt;").replace(">","&gt;").replace("\n","<br>")

def _ext_icon(fname):
    ext=pathlib.Path(str(fname)).suffix.lower()
    return {".pdf":"P",".csv":"C",".xlsx":"X",".xls":"X",
            ".jpg":"I",".jpeg":"I",".png":"I",".txt":"T",".docx":"D"}.get(ext,"F")

def _safe(v, default=""):
    """Coerce any value to str, replacing None with default."""
    if v is None: return default
    return str(v)

def _new_session(name=None):
    sid=name or f"Chat {uuid.uuid4().hex[:4].upper()}"
    return {"id":sid,"thread":str(uuid.uuid4()),"history":[],
            "files":{},"pipeline":None,"mode":"Auto"}

def _cfg(s): return {"configurable":{"thread_id":s["thread"]}}
def _new_app():
    s=_new_session("Chat 1")
    return {"sessions":{s["id"]:s},"active":s["id"]}
def _active(app): return app["sessions"][app["active"]]
def _upsert(app,sess):
    app=dict(app);smap=dict(app["sessions"]);smap[sess["id"]]=sess;app["sessions"]=smap;return app
def _set_active(app,sid):
    app=dict(app);app["active"]=sid;return app

def _chart_iframe(chart_html):
    ch = _safe(chart_html)
    if not ch.strip(): return ""
    b64=base64.b64encode(ch.encode("utf-8")).decode("ascii")
    return (f'<iframe class="chart-frame" src="data:text/html;base64,{b64}" '
            f'style="width:100%;min-height:360px;border:none;border-radius:6px;'
            f'margin-top:10px;display:block;background:#fff;"></iframe>')

def _preload_select_html():
    opts='<option value="" disabled selected style="color:#9c9890;">↓ Preload a document…</option>'
    for name in PRELOAD_CATALOG:
        opts+=f'<option value="{_e(name)}" style="color:#1c1a17;background:#fff;">{_e(name)}</option>'
    return (f'<select id="preload-native-sel" '
            f'style="flex:1;width:100%;height:30px;background:#edeae4;color:#1c1a17;'
            f'-webkit-text-fill-color:#1c1a17;border:1.5px solid #c8c3b8;border-radius:5px;'
            f'font-family:DM Sans,system-ui,sans-serif;font-size:11px;font-weight:500;'
            f'padding:0 8px;cursor:pointer;outline:none;-webkit-appearance:auto;appearance:auto;">'
            f'{opts}</select>')

# ── render ─────────────────────────────────────────────────────────────────
def _render(app):
    sessions=app["sessions"]; active_id=app["active"]
    sess=sessions[active_id]
    history=sess.get("history",[]); files=sess.get("files",{})
    mode=sess.get("mode","Auto")
    turns=sum(1 for m in history if m.get("role")=="me")

    sess_html=""
    for sid,s in sessions.items():
        t=sum(1 for m in s.get("history",[]) if m.get("role")=="me")
        fc=len(s.get("files",{}))
        ac=" active" if sid==active_id else ""
        ftag=f'<span class="sc-tag">{fc}f</span>' if fc else ""
        safe_sid=sid.replace("\\","\\\\").replace("'","\\'")
        sess_html+=(f'<div class="sess-chip{ac}" onclick="window.__pendingSid=\'{safe_sid}\';">'
                    f'<span class="sc-name">{_e(sid)}</span>'
                    f'<div class="sc-meta"><span>{t}t</span>{ftag}</div></div>')

    fhtml=""
    for fname,rec in files.items():
        stem=pathlib.Path(fname).stem[:20]; rtype=_safe(rec.get("source_type"),"")
        fhtml+=(f'<div class="file-chip"><div class="fc-row">'
                f'<div class="fc-icon">{_ext_icon(fname)}</div>'
                f'<span class="fc-name">{_e(stem)}</span></div>'
                f'<span class="fc-type">{_e(rtype)}</span></div>')
    if not fhtml: fhtml='<div class="no-items">No files in this session</div>'

    fpills="".join(
        f'<span class="tpill tpf">{_ext_icon(fn)} {_e(pathlib.Path(fn).stem[:10])}</span>'
        for fn in list(files.keys())[:2])

    if not history:
        msgs=('<div class="empty-wrap">'
              '<div class="empty-glyph">◇</div>'
              '<p class="empty-t">Upload or preload a document,<br>then ask anything about it.</p>'
              '<span class="empty-h">PDF · CSV · Excel · Image · TXT</span></div>')
    else:
        msgs=""
        for m in history:
            role=m.get("role","ai"); av="YOU" if role=="me" else "AI"
            ftags="".join(f'<span class="ftag">↳ {_e(f)}</span>' for f in (m.get("files") or []))
            ms_=m.get("ms"); rt=_safe(m.get("route")); meta=""
            if ms_ is not None or rt:
                meta='<div class="meta">'
                if rt: meta+=f'<span class="tpill tpr">→ {_e(rt)}</span>'
                if ms_ is not None: meta+=f'<span class="tpill tpms">{ms_:.0f}ms</span>'
                meta+="</div>"
            text_body  = _safe(m.get("text"))
            chart_html = _safe(m.get("chart_html"))
            is_html    = role=="ai" and bool(chart_html.strip())
            chart_r    = _chart_iframe(chart_html) if is_html else ""
            avcls="ai" if role=="ai" else "me"
            msgs+=(f'<div class="msg {role}"><div class="mav {avcls}">{av}</div>'
                   f'<div class="bbl">{_e(text_body)}{chart_r}{ftags}{meta}</div></div>')

    return f"""
<div id="idoc-page">
  <div id="idoc-hdr">
    <div class="logo"><div class="logo-mark"></div>
    <div class="logo-tx">Intelli<em>Doc</em></div></div>
    <div class="hsp"></div><span class="ver">v14</span><div class="sdot"></div>
  </div>
  <div id="idoc-body">
    <div id="idoc-sb">
      <div class="sb-sec">Sessions</div>
      <div class="sess-scroll">{sess_html}</div>
      <div class="sb-sec">Documents</div>
      <div class="file-scroll">{fhtml}</div>
      <div class="sb-foot"><div class="uav">U</div><div class="ulbl">{_e(mode)}</div></div>
    </div>
    <div id="idoc-main">
      <div id="idoc-topbar">
        <span class="sname">{_e(active_id)}</span>
        <span class="ssep">/</span><span class="sturns">{turns}t</span>
        <div class="hsp"></div>{fpills}
      </div>
      <div id="idoc-chat">{msgs}</div>
    </div>
  </div>
</div>
{_RESIZE_JS}"""

# ── Gradio app ─────────────────────────────────────────────────────────────
with gr.Blocks(theme=gr.themes.Base(), css=_CSS, title="IntelliDoc AI") as demo:

    app_state   = gr.State(value=_new_app())
    chat_out    = gr.HTML(elem_id="idoc-chat-wrap", value="")
    sw_inp      = gr.Textbox(value="", visible=False, elem_id="sw-inp",      interactive=True)
    preload_inp = gr.Textbox(value="", visible=False, elem_id="preload-inp", interactive=True)

    with gr.Column(elem_id="idoc-bottom"):
        status_md = gr.Markdown("", elem_id="idoc-status")

        with gr.Row(elem_id="idoc-chips", equal_height=True):
            mode_radio = gr.Radio(
                choices=list(_MODE_MAP.keys()), value="Auto",
                label="", interactive=True)

        with gr.Row(elem_id="idoc-actions", equal_height=True):
            # Native <select> as gr.HTML — lives in the same Row as buttons
            sel_html     = gr.HTML(_preload_select_html(), elem_id="idoc-sel-wrap")
            preload_btn  = gr.Button("↓ Load",    scale=1, min_width=64)
            upload_btn   = gr.UploadButton(
                "↑ Upload", scale=1, min_width=64,
                file_types=[".pdf",".csv",".xlsx",".xls",".txt",".docx",".png",".jpg"],
                file_count="multiple", elem_classes=["upload-btn"])
            new_sess_btn = gr.Button("+ Session", scale=1, min_width=74)

        with gr.Row(elem_id="idoc-inp", equal_height=False):
            txt_in = gr.Textbox(
                placeholder="Ask anything about your document…",
                lines=1, max_lines=2,
                show_label=False, container=False, scale=9)
            with gr.Column(scale=0, min_width=74, elem_id="send-btn"):
                send_btn = gr.Button("Send →", variant="primary")

    # ── callbacks ──────────────────────────────────────────────────────────
    demo.load(fn=lambda a: _render(a), inputs=[app_state], outputs=[chat_out])

    def on_switch(sid, app):
        sid=(sid or "").strip()
        if sid and sid in app["sessions"]:
            app=_set_active(app,sid)
        sess=_active(app)
        return app, _render(app), "", sess.get("mode","Auto")
    sw_inp.change(fn=on_switch, inputs=[sw_inp,app_state],
                  outputs=[app_state,chat_out,sw_inp,mode_radio])

    def on_mode(mode,app):
        sess=dict(_active(app)); sess["mode"]=mode
        app=_upsert(app,sess)
        return app,_render(app),f"mode → {mode}"
    mode_radio.change(fn=on_mode,inputs=[mode_radio,app_state],
                      outputs=[app_state,chat_out,status_md])

    def on_new(app):
        existing=set(app["sessions"].keys()); i=len(existing)+1
        while f"Chat {i}" in existing: i+=1
        ns=_new_session(f"Chat {i}")
        app=_upsert(app,ns); app=_set_active(app,ns["id"])
        return app,_render(app),"",f"new: {ns['id']}"
    new_sess_btn.click(fn=on_new,inputs=[app_state],
                       outputs=[app_state,chat_out,txt_in,status_md])

    def on_preload(fname,app):
        fname=_safe(fname).strip()
        if not fname: return app,_render(app),"⚠ select a file from the dropdown first"
        fpath=PRELOAD_CATALOG.get(fname,"")
        if not fpath or not pathlib.Path(fpath).exists():
            return app,_render(app),f"✗ not found: {fname}"
        sess=dict(_active(app)); sess["files"]=dict(sess.get("files",{}))
        try:
            r=_ingest(fpath)
            # FIX: coerce all _ingest return values — source_type may be None
            source_type = _safe(r.get("source_type"), "TEXT_RAG") or "TEXT_RAG"
            p=_build(
                rag_index  = r.get("rag_index"),
                doc_index  = r.get("doc_index"),
                table_names= r.get("sql_tables"),
                source_type= source_type)
            sess["files"][fname]={"path":fpath,"source_type":source_type}
            sess["pipeline"]=p; app=_upsert(app,sess)
            return app,_render(app),f"✓ {fname}"
        except Exception as e:
            import traceback; traceback.print_exc()
            return app,_render(app),f"✗ {e}"

    # Triggered by JS onchange (immediate) OR Load button click
    preload_inp.change(fn=on_preload,inputs=[preload_inp,app_state],
                       outputs=[app_state,chat_out,status_md])
    preload_btn.click(fn=on_preload,inputs=[preload_inp,app_state],
                      outputs=[app_state,chat_out,status_md])

    def on_upload(files,app):
        if not files: return app,_render(app),"⚠ no files"
        sess=dict(_active(app)); sess["files"]=dict(sess.get("files",{}))
        loaded=[]; last_p=sess.get("pipeline")
        for f in (files if isinstance(files,list) else [files]):
            fpath=f.name if hasattr(f,"name") else str(f)
            fname=pathlib.Path(fpath).name
            try:
                r=_ingest(fpath)
                # FIX: coerce source_type — never let None reach _build or dict
                source_type = _safe(r.get("source_type"), "TEXT_RAG") or "TEXT_RAG"
                p=_build(
                    rag_index  = r.get("rag_index"),
                    doc_index  = r.get("doc_index"),
                    table_names= r.get("sql_tables"),
                    source_type= source_type)
                sess["files"][fname]={"path":fpath,"source_type":source_type}
                last_p=p; loaded.append(fname)
            except Exception as e:
                import traceback; traceback.print_exc()
                return app,_render(app),f"✗ {fname}: {e}"
        sess["pipeline"]=last_p; app=_upsert(app,sess)
        return app,_render(app),f"✓ {', '.join(loaded)}"
    upload_btn.upload(fn=on_upload,inputs=[upload_btn,app_state],
                      outputs=[app_state,chat_out,status_md])

    def on_send(question,app):
        question=_safe(question).strip()
        if not question: return app,_render(app),"","⚠ type something"
        sess=_active(app)
        if not sess.get("pipeline"): return app,_render(app),"","⚠ load a file first"
        sess=dict(sess); history=list(sess.get("history",[])); files=sess.get("files",{})
        history.append({"role":"me","text":question})
        sess["history"]=history
        t0=time.perf_counter()
        try:
            force_route=_MODE_MAP.get(sess.get("mode","Auto"))  # None = Auto, that's fine
            res=sess["pipeline"].invoke(
                {"user_query":question,
                 "_thread_id":sess["thread"],
                 "file_names":list(files.keys()),
                 "force_route":force_route,
                 "chart_html":""},
                _cfg(sess))
            ans        = _safe(res.get("final_response"))
            route      = _safe(res.get("route"))
            chart_html = _safe(res.get("chart_html"))
            is_html    = bool(chart_html.strip())
        except Exception as e:
            import traceback; traceback.print_exc()
            ans="⚠ "+str(e); route="error"; chart_html=""; is_html=False
        ms=(time.perf_counter()-t0)*1000
        history.append({"role":"ai","text":ans,"chart_html":chart_html,
                         "ms":ms,"route":route,"is_html":is_html})
        sess["history"]=history; app=_upsert(app,sess)
        return app,_render(app),"",f"→ {route}  ·  {ms:.0f}ms"

    send_btn.click(fn=on_send,inputs=[txt_in,app_state],
                   outputs=[app_state,chat_out,txt_in,status_md])
    txt_in.submit(fn=on_send,inputs=[txt_in,app_state],
                  outputs=[app_state,chat_out,txt_in,status_md])

# demo.queue().launch(inline=True)

In [ ]:
"""
IntelliDoc AI — Gradio UI v15  (Gradio 3.50.2)
===============================================
FIXES vs v14:
1. DROPDOWN + LOAD BUTTON: replaced native <select>+hidden-textbox+JS bridge
   with a real gr.Dropdown.  Load button reads it directly — zero JS needed.
2. BM25 ZeroDivisionError: guard added before BM25Okapi() — raises a friendly
   error when all chunks are deduped away (image-only / blank PDFs).
3. SQL CHART IN BUBBLE: _render() now checks chart_html directly instead of
   the stored is_html flag which could be stale/False.
4. UPLOAD wires register_files() so the pipeline has the dataframe reference
   exactly like the inline tester does.
"""

import os, uuid, time, pathlib, base64
import gradio as gr

# ── shims (replaced by real functions when the other cells run first) ───────
try:
    _ingest          = ingest_document    # noqa: F821
    _build           = build_pipeline     # noqa: F821
    _register        = register_files     # noqa: F821
except NameError:
    def _ingest(path):
        return {"rag_index": None, "doc_index": None,
                "sql_tables": [], "source_type": "TEXT_RAG"}
    def _build(rag_index=None, doc_index=None, table_names=None, source_type=None):
        class _D:
            def invoke(self, inp, config=None):
                q = inp.get("user_query", "")
                return {"final_response": f"[Demo] You asked: {q}",
                        "route": source_type or "TEXT_RAG", "chart_html": ""}
        return _D()
    def _register(thread_id, files_dict):
        pass

PRELOAD_CATALOG: dict = {
    "digital PDF.pdf": "/kaggle/input/datasets/dp0741470/document/RAG doc.pdf",
    "OCR image.jpg":   "/kaggle/input/datasets/devanshipatel05/ocr-samples/0000114117.jpg",
    "CSV sheet":       "/kaggle/input/datasets/dp0741470/excelsheet/Sample - Superstore.csv",
}

_MODE_MAP = {
    "Auto":       None,
    "Text RAG":   "TEXT_RAG",
    "SQL / Data": "SQL",
    "Visual RAG": "IMAGE_RAG",
}

# ── CSS ─────────────────────────────────────────────────────────────────────
_CSS = """
@import url('https://fonts.googleapis.com/css2?family=Geist+Mono:wght@400;500&family=DM+Sans:opsz,wght@9..40,300;9..40,400;9..40,500;9..40,600&display=swap');
*,*::before,*::after{box-sizing:border-box;margin:0;padding:0}
:root{
  --bg:#f4f2ee;--bg2:#ffffff;--bg3:#edeae4;--bg4:#e5e1d8;
  --bd:#ddd9d0;--bd2:#c8c3b8;
  --tx:#1c1a17;--tx2:#5c5750;--tx3:#9c9890;
  --acc:#7c5c2e;--acc2:#a07840;--accbg:#f5ede0;
  --grn:#1a7a4a;--grnbg:#e6f5ee;--gnbd:#b8dfc8;
  --blu:#2d5fa8;--blubg:#e6eef8;--blubd:#b8ccee;
  --r:8px;--f:'DM Sans',system-ui,sans-serif;--m:'Geist Mono',monospace;
  --sh:0 1px 3px rgba(0,0,0,0.07);--sh2:0 2px 8px rgba(0,0,0,0.12)
}
html,body{background:var(--bg)!important}
.gradio-container{width:100%!important;max-width:100%!important;padding:0!important;margin:0!important;background:var(--bg)!important;font-family:var(--f)!important}
.gradio-container>.contain,.gradio-container>.contain>div,.gradio-container>.contain>div>div{padding:0!important;gap:0!important;background:var(--bg)!important}
#component-0{padding:0!important;gap:0!important;background:transparent!important;border:none!important;box-shadow:none!important}
.gr-block,.gr-box,.gr-form{border:none!important;background:transparent!important;box-shadow:none!important;padding:0!important;margin:0!important}
footer{display:none!important}.gr-prose{display:none!important}

#idoc-page{display:flex;flex-direction:column;width:100%;background:var(--bg);font-family:var(--f);color:var(--tx)}
#idoc-hdr{height:46px;flex-shrink:0;background:var(--bg2);border-bottom:1px solid var(--bd);padding:0 18px;display:flex;align-items:center;gap:12px;z-index:20;box-shadow:var(--sh)}
.logo{display:flex;align-items:center;gap:8px}
.logo-mark{width:20px;height:20px;border-radius:5px;border:1.5px solid var(--acc2);background:var(--accbg);display:flex;align-items:center;justify-content:center}
.logo-mark::after{content:'';width:6px;height:6px;background:var(--acc);border-radius:1px;transform:rotate(45deg);display:block}
.logo-tx{font-size:14px;font-weight:600;color:var(--tx);letter-spacing:-.04em}
.logo-tx em{font-style:normal;color:var(--acc)}
.hsp{flex:1}
.ver{font-family:var(--m);font-size:9px;color:var(--tx3);padding:2px 7px;border:1px solid var(--bd);border-radius:4px;background:var(--bg3)}
.sdot{width:6px;height:6px;border-radius:50%;background:var(--grn);box-shadow:0 0 0 2px rgba(26,122,74,.15)}
#idoc-body{display:flex;flex:1;overflow:hidden;min-height:0}
#idoc-sb{width:196px;flex-shrink:0;background:var(--bg2);border-right:1px solid var(--bd);display:flex;flex-direction:column;overflow:hidden}
.sb-sec{flex-shrink:0;padding:10px 12px 5px;font-size:8px;font-weight:600;color:var(--tx3);letter-spacing:.14em;text-transform:uppercase;font-family:var(--m)}
.sess-scroll{overflow-y:auto;padding:0 8px 6px;max-height:200px;min-height:60px;border-bottom:1px solid var(--bd);flex-shrink:0}
.sess-scroll::-webkit-scrollbar{width:3px}
.sess-scroll::-webkit-scrollbar-thumb{background:var(--bd2);border-radius:3px}
.sess-chip{padding:7px 9px;border-radius:var(--r);margin-bottom:3px;background:var(--bg3);border:1px solid var(--bd);cursor:pointer;transition:all .12s;display:flex;flex-direction:column;gap:2px;box-shadow:0 1px 3px rgba(0,0,0,.06)}
.sess-chip:hover{border-color:var(--bd2);background:var(--bg4);box-shadow:var(--sh2)}
.sess-chip.active{background:var(--accbg);border-color:var(--acc2)}
.sc-name{font-size:11px;font-weight:500;color:var(--tx);white-space:nowrap;overflow:hidden;text-overflow:ellipsis}
.sess-chip.active .sc-name{color:var(--acc)}
.sc-meta{display:flex;gap:5px;align-items:center;font-size:8.5px;color:var(--tx3);font-family:var(--m)}
.sc-tag{background:var(--bg4);border:1px solid var(--bd);border-radius:3px;padding:0 4px;font-size:7.5px}
.file-scroll{flex:1;overflow-y:auto;padding:0 8px 8px}
.file-chip{padding:7px 9px;border-radius:var(--r);margin-bottom:3px;background:var(--bg3);border:1px solid var(--bd);display:flex;flex-direction:column;gap:3px}
.fc-row{display:flex;align-items:center;gap:6px}
.fc-icon{width:14px;height:14px;border-radius:3px;background:var(--accbg);border:1px solid var(--bd2);display:flex;align-items:center;justify-content:center;font-size:7.5px;color:var(--acc);font-family:var(--m);font-weight:600;flex-shrink:0}
.fc-name{font-size:10.5px;font-weight:500;color:var(--tx2);white-space:nowrap;overflow:hidden;text-overflow:ellipsis}
.fc-type{font-size:8px;color:var(--tx3);font-family:var(--m);padding-left:20px}
.no-items{font-size:11px;color:var(--tx3);padding:10px 6px;text-align:center;line-height:1.7;font-style:italic}
.sb-foot{flex-shrink:0;padding:8px 12px;border-top:1px solid var(--bd);display:flex;align-items:center;gap:8px}
.uav{width:20px;height:20px;border-radius:50%;background:var(--accbg);border:1px solid var(--bd2);display:flex;align-items:center;justify-content:center;font-size:7.5px;font-weight:700;color:var(--acc);font-family:var(--m)}
.ulbl{font-size:10px;color:var(--tx3)}
#idoc-main{flex:1;display:flex;flex-direction:column;overflow:hidden;min-width:0}
#idoc-topbar{height:38px;flex-shrink:0;background:var(--bg2);border-bottom:1px solid var(--bd);padding:0 16px;display:flex;align-items:center;gap:9px}
.sname{font-size:12px;font-weight:500;color:var(--tx);letter-spacing:-.02em}
.ssep{color:var(--tx3);font-size:10px}
.sturns{font-size:9.5px;color:var(--tx3);font-family:var(--m)}
.tpill{display:inline-flex;align-items:center;gap:3px;padding:2px 7px;border-radius:4px;font-size:9px;font-family:var(--m);border:1px solid transparent}
.tpf{background:var(--accbg);color:var(--acc);border-color:var(--bd2)}
.tpr{background:var(--grnbg);color:var(--grn);border-color:var(--gnbd)}
.tpms{background:var(--blubg);color:var(--blu);border-color:var(--blubd)}
#idoc-chat{overflow-y:scroll;overflow-x:hidden;padding:18px 20px;display:flex;flex-direction:column;gap:13px;scroll-behavior:smooth;background:var(--bg);min-height:300px;flex:1}
#idoc-chat::-webkit-scrollbar{width:5px}
#idoc-chat::-webkit-scrollbar-thumb{background:var(--bd2);border-radius:3px}
.empty-wrap{flex:1;display:flex;flex-direction:column;align-items:center;justify-content:center;gap:12px;text-align:center;padding:60px 20px}
.empty-glyph{width:40px;height:40px;border-radius:var(--r);border:1px solid var(--bd2);background:var(--bg2);display:flex;align-items:center;justify-content:center;color:var(--tx3);font-size:18px;box-shadow:var(--sh)}
.empty-t{font-size:12.5px;color:var(--tx2);line-height:1.65}
.empty-h{font-size:9.5px;color:var(--tx3);font-family:var(--m);letter-spacing:.04em}
.msg{display:flex;gap:9px;align-items:flex-start;animation:mi .16s ease}
@keyframes mi{from{opacity:0;transform:translateY(4px)}to{opacity:1;transform:none}}
.msg.me{flex-direction:row-reverse}
.mav{width:22px;height:22px;min-width:22px;border-radius:5px;display:flex;align-items:center;justify-content:center;font-size:7px;font-weight:700;font-family:var(--m);flex-shrink:0;margin-top:2px}
.mav.ai{background:var(--bg3);border:1px solid var(--bd2);color:var(--tx2)}
.mav.me{background:var(--accbg);border:1px solid var(--bd2);color:var(--acc)}
.bbl{max-width:74%;padding:10px 13px;font-size:12.5px;line-height:1.75;border-radius:var(--r)}
.msg.me .bbl{background:var(--bg2);border:1px solid var(--bd);border-radius:var(--r) var(--r) 3px var(--r);color:var(--tx);box-shadow:var(--sh)}
.msg.ai .bbl{background:var(--accbg);border:1px solid var(--bd);border-radius:3px var(--r) var(--r) var(--r);color:var(--tx);box-shadow:var(--sh);max-width:82%}
.meta{display:flex;gap:5px;flex-wrap:wrap;margin-top:8px;padding-top:8px;border-top:1px solid var(--bd)}
.ftag{display:inline-flex;align-items:center;gap:3px;padding:2px 6px;border-radius:4px;font-size:9px;font-family:var(--m);background:var(--grnbg);color:var(--grn);border:1px solid var(--gnbd)}
.chart-frame{width:100%;min-height:360px;border:none;border-radius:6px;margin-top:10px;display:block;background:#fff}
#idoc-bottom{flex-shrink:0;background:var(--bg2)!important;border-top:1px solid var(--bd);padding:8px 14px 10px;display:flex;flex-direction:column;gap:5px;z-index:10}
#idoc-status{font-family:var(--m);font-size:10px;color:var(--tx2);line-height:1.4;min-height:16px}
#idoc-status>div,#idoc-status p{margin:0!important;padding:0!important;background:transparent!important;color:var(--tx2)!important;font-size:10px!important}
#idoc-chips{display:flex!important;align-items:center!important;gap:4px!important;flex-shrink:0;height:28px;background:transparent!important}
#idoc-chips,#idoc-chips>*,#idoc-chips .gr-form,#idoc-chips .gr-box,#idoc-chips fieldset,#idoc-chips .block,#idoc-chips .gr-block,#idoc-chips>div,#idoc-chips>div>div{background:transparent!important;border:none!important;box-shadow:none!important;padding:0!important;margin:0!important;height:auto!important}
#idoc-chips fieldset>div,#idoc-chips .gr-radio-row{display:flex!important;gap:4px!important;flex-wrap:nowrap!important;align-items:center!important;background:transparent!important}
#idoc-chips label{display:inline-flex!important;align-items:center!important;justify-content:center!important;padding:0 10px!important;height:26px!important;border-radius:5px!important;border:1px solid var(--bd)!important;background:var(--bg3)!important;color:var(--tx)!important;font-size:10.5px!important;font-weight:500!important;cursor:pointer!important;font-family:var(--f)!important;transition:all .1s!important;white-space:nowrap!important;line-height:1!important;user-select:none!important;box-shadow:0 1px 3px rgba(0,0,0,.08)!important}
#idoc-chips label>span{display:inline!important;font-size:10.5px!important;color:inherit!important;pointer-events:none!important}
#idoc-chips label:has(input:checked){background:var(--accbg)!important;color:var(--acc)!important;border-color:var(--acc2)!important;font-weight:600!important}
#idoc-chips label:hover:not(:has(input:checked)){background:var(--bg4)!important;color:var(--tx)!important;border-color:var(--bd2)!important}
#idoc-chips label input[type=radio]{display:none!important}
#idoc-chips .label-wrap,#idoc-chips>.block>span{display:none!important}
#idoc-actions{display:flex!important;align-items:center!important;gap:5px!important;flex-shrink:0;height:32px;background:transparent!important}
#idoc-actions,#idoc-actions>*,#idoc-actions .gr-form,#idoc-actions .gr-box,#idoc-actions .block,#idoc-actions .gr-block{background:transparent!important;border:none!important;box-shadow:none!important;padding:0!important;margin:0!important}
#idoc-actions .gr-block>label,#idoc-actions .block>label,#idoc-actions .label-wrap{display:none!important}

/* ── Preload Dropdown (gr.Dropdown) styled to match toolbar ── */
#preload-dd{flex:1!important;min-width:0!important;max-width:none!important}
#preload-dd .wrap,#preload-dd .wrap-inner,#preload-dd input{
  height:30px!important;min-height:30px!important;max-height:30px!important;
  background:#edeae4!important;color:#1c1a17!important;
  border:1.5px solid #c8c3b8!important;border-radius:5px!important;
  font-family:'DM Sans',system-ui,sans-serif!important;
  font-size:11px!important;font-weight:500!important;
  padding:0 8px!important;box-shadow:0 1px 3px rgba(0,0,0,.08)!important}
#preload-dd .wrap:hover,#preload-dd:hover .wrap{border-color:#a07840!important}
#preload-dd input{cursor:pointer!important}
#preload-dd input::placeholder{color:#9c9890!important;font-size:11px!important}
#preload-dd ul.options{background:#fff!important;border:1.5px solid #c8c3b8!important;
  border-radius:5px!important;margin-top:2px!important;z-index:999!important;
  box-shadow:0 4px 12px rgba(0,0,0,.12)!important}
#preload-dd ul.options li{font-size:11px!important;color:#1c1a17!important;
  padding:6px 10px!important;font-family:'DM Sans',system-ui,sans-serif!important}
#preload-dd ul.options li:hover,#preload-dd ul.options li.selected{
  background:#f5ede0!important;color:#7c5c2e!important}
#preload-dd .label-wrap,#preload-dd>label,#preload-dd .block>span{display:none!important}
#preload-dd .secondary-wrap svg{width:12px!important;height:12px!important;
  color:#9c9890!important}
#preload-dd .token{display:none!important}

#idoc-actions button,#idoc-actions .upload-btn button,#idoc-actions .gr-block button,#idoc-actions .block button{height:30px!important;padding:0 12px!important;background:var(--bg3)!important;color:var(--tx)!important;border:1.5px solid var(--bd2)!important;border-radius:5px!important;font-family:var(--f)!important;font-size:11px!important;font-weight:500!important;white-space:nowrap!important;cursor:pointer!important;transition:all .1s!important;flex-shrink:0!important;box-shadow:0 1px 4px rgba(0,0,0,.10)!important}
#idoc-actions button:hover,#idoc-actions .upload-btn button:hover{background:var(--bg4)!important;color:var(--acc)!important;border-color:var(--acc2)!important}
#idoc-inp{display:flex!important;gap:7px!important;align-items:flex-end!important;flex-shrink:0;background:transparent!important}
#idoc-inp,#idoc-inp>*,#idoc-inp .gr-block,#idoc-inp .gr-box,#idoc-inp .gr-text-input,#idoc-inp .block{background:transparent!important;border:none!important;box-shadow:none!important;padding:0!important;margin:0!important}
#idoc-inp>.gr-block,#idoc-inp>.block{flex:1!important;min-width:0!important}
#idoc-inp .label-wrap{display:none!important}
#idoc-inp textarea,#idoc-inp input[type=text]{width:100%!important;display:block!important;background:var(--bg3)!important;color:var(--tx)!important;border:1.5px solid var(--bd)!important;border-radius:7px!important;font-family:var(--f)!important;font-size:13px!important;padding:8px 12px!important;resize:none!important;outline:none!important;min-height:36px!important;max-height:64px!important;overflow-y:auto!important;line-height:1.55!important;caret-color:var(--acc)!important;transition:border-color .12s,box-shadow .12s!important}
#idoc-inp textarea:focus,#idoc-inp input[type=text]:focus{border-color:var(--acc2)!important;box-shadow:0 0 0 3px rgba(160,120,64,.1)!important;background:var(--bg2)!important}
#idoc-inp textarea::placeholder{color:var(--tx3)!important;font-size:12px!important}
#send-btn{flex-shrink:0!important}
#send-btn button{background:var(--acc)!important;color:#fff!important;border:none!important;border-radius:7px!important;font-family:var(--f)!important;font-size:12px!important;font-weight:600!important;height:36px!important;padding:0 16px!important;white-space:nowrap!important;cursor:pointer!important;transition:all .1s!important;box-shadow:0 2px 6px rgba(124,92,46,.25)!important}
#send-btn button:hover{background:#6a4e26!important;transform:translateY(-1px)!important}
#idoc-bottom .block,#idoc-bottom .gr-block,#idoc-bottom .gr-box,#idoc-bottom .gr-form{background:transparent!important;border:none!important;box-shadow:none!important;padding:0!important;margin:0!important}
"""

# ── JS (layout + session switching only — no preload JS needed anymore) ──────
_RESIZE_JS = """
<script>
(function(){
  /* ── Layout sizing ── */
  function layout(){
    var page=document.getElementById('idoc-page');
    var chat=document.getElementById('idoc-chat');
    var hdr=document.getElementById('idoc-hdr');
    var topbar=document.getElementById('idoc-topbar');
    var bottom=document.getElementById('idoc-bottom');
    if(!page||!chat||!hdr||!topbar||!bottom)return;
    var c=page.parentElement;
    while(c&&c.offsetHeight<200)c=c.parentElement;
    var total=c?c.offsetHeight:window.innerHeight;
    page.style.height=total+'px';
    document.getElementById('idoc-body').style.height=(total-hdr.offsetHeight)+'px';
    chat.style.height=Math.max(
      total-hdr.offsetHeight-topbar.offsetHeight-bottom.offsetHeight-2, 300)+'px';
    chat.scrollTop=chat.scrollHeight+9999;
  }
  layout();
  window.addEventListener('resize', layout);
  if(window.ResizeObserver)
    new ResizeObserver(layout).observe(
      document.querySelector('.gradio-container')||document.body);
  [100,500,1500].forEach(function(t){ setTimeout(layout,t); });

  /* ── helper: set a Gradio 3.x hidden textbox value ── */
  function setGradioText(selector, val){
    var el=document.querySelector(selector);
    if(!el)return false;
    var s=Object.getOwnPropertyDescriptor(window.HTMLTextAreaElement.prototype,'value')||
           Object.getOwnPropertyDescriptor(window.HTMLInputElement.prototype,'value');
    if(s&&s.set)s.set.call(el,val); else el.value=val;
    el.dispatchEvent(new Event('input',{bubbles:true}));
    el.dispatchEvent(new Event('change',{bubbles:true}));
    return true;
  }

  /* ── Session switch ── */
  window.__pendingSid=null;
  setInterval(function(){
    if(!window.__pendingSid)return;
    var sid=window.__pendingSid; window.__pendingSid=null;
    setGradioText('#sw-inp textarea,#sw-inp input', sid);
  }, 120);
})();
</script>
"""

# ── helpers ──────────────────────────────────────────────────────────────────
def _e(t):
    return str(t).replace("&","&amp;").replace("<","&lt;").replace(">","&gt;").replace("\n","<br>")

def _ext_icon(fname):
    ext = pathlib.Path(str(fname)).suffix.lower()
    return {".pdf":"P",".csv":"C",".xlsx":"X",".xls":"X",
            ".jpg":"I",".jpeg":"I",".png":"I",".txt":"T",".docx":"D"}.get(ext,"F")

def _safe(v, default=""):
    """Coerce any value to str, replacing None with default."""
    if v is None: return default
    return str(v)

def _new_session(name=None):
    sid = name or f"Chat {uuid.uuid4().hex[:4].upper()}"
    return {"id":sid, "thread":str(uuid.uuid4()), "history":[],
            "files":{}, "pipeline":None, "mode":"Auto"}

def _cfg(s):   return {"configurable": {"thread_id": s["thread"]}}
def _new_app():
    s = _new_session("Chat 1")
    return {"sessions":{s["id"]:s}, "active":s["id"]}
def _active(app):   return app["sessions"][app["active"]]
def _upsert(app, sess):
    app=dict(app); smap=dict(app["sessions"]); smap[sess["id"]]=sess
    app["sessions"]=smap; return app
def _set_active(app, sid):
    app=dict(app); app["active"]=sid; return app

def _chart_iframe(chart_html):
    ch = _safe(chart_html)
    if not ch.strip(): return ""
    b64 = base64.b64encode(ch.encode("utf-8")).decode("ascii")
    return (f'<iframe class="chart-frame" src="data:text/html;base64,{b64}" '
            f'style="width:100%;min-height:360px;border:none;border-radius:6px;'
            f'margin-top:10px;display:block;background:#fff;"></iframe>')

# ── render ───────────────────────────────────────────────────────────────────
def _render(app):
    sessions  = app["sessions"]; active_id = app["active"]
    sess      = sessions[active_id]
    history   = sess.get("history", []); files = sess.get("files", {})
    mode      = sess.get("mode", "Auto")
    turns     = sum(1 for m in history if m.get("role") == "me")

    sess_html = ""
    for sid, s in sessions.items():
        t  = sum(1 for m in s.get("history",[]) if m.get("role")=="me")
        fc = len(s.get("files", {}))
        ac = " active" if sid == active_id else ""
        ftag = f'<span class="sc-tag">{fc}f</span>' if fc else ""
        safe_sid = sid.replace("\\","\\\\").replace("'","\\'")
        sess_html += (f'<div class="sess-chip{ac}" onclick="window.__pendingSid=\'{safe_sid}\';">'
                      f'<span class="sc-name">{_e(sid)}</span>'
                      f'<div class="sc-meta"><span>{t}t</span>{ftag}</div></div>')

    fhtml = ""
    for fname, rec in files.items():
        stem  = pathlib.Path(fname).stem[:20]
        rtype = _safe(rec.get("source_type"), "")
        fhtml += (f'<div class="file-chip"><div class="fc-row">'
                  f'<div class="fc-icon">{_ext_icon(fname)}</div>'
                  f'<span class="fc-name">{_e(stem)}</span></div>'
                  f'<span class="fc-type">{_e(rtype)}</span></div>')
    if not fhtml:
        fhtml = '<div class="no-items">No files in this session</div>'

    fpills = "".join(
        f'<span class="tpill tpf">{_ext_icon(fn)} {_e(pathlib.Path(fn).stem[:10])}</span>'
        for fn in list(files.keys())[:2])

    if not history:
        msgs = ('<div class="empty-wrap">'
                '<div class="empty-glyph">◇</div>'
                '<p class="empty-t">Upload or preload a document,<br>then ask anything about it.</p>'
                '<span class="empty-h">PDF · CSV · Excel · Image · TXT</span></div>')
    else:
        msgs = ""
        for m in history:
            role  = m.get("role", "ai")
            av    = "YOU" if role == "me" else "AI"
            ftags = "".join(f'<span class="ftag">↳ {_e(f)}</span>'
                            for f in (m.get("files") or []))
            ms_   = m.get("ms"); rt = _safe(m.get("route")); meta = ""
            if ms_ is not None or rt:
                meta = '<div class="meta">'
                if rt:           meta += f'<span class="tpill tpr">→ {_e(rt)}</span>'
                if ms_ is not None: meta += f'<span class="tpill tpms">{ms_:.0f}ms</span>'
                meta += "</div>"
            text_body  = _safe(m.get("text"))
            chart_html = _safe(m.get("chart_html"))
            # FIX 3: always render chart when chart_html is non-empty —
            #         never rely on the stored is_html flag
            chart_r    = _chart_iframe(chart_html) if chart_html.strip() else ""
            avcls      = "ai" if role == "ai" else "me"
            msgs += (f'<div class="msg {role}"><div class="mav {avcls}">{av}</div>'
                     f'<div class="bbl">{_e(text_body)}{chart_r}{ftags}{meta}</div></div>')

    return f"""
<div id="idoc-page">
  <div id="idoc-hdr">
    <div class="logo"><div class="logo-mark"></div>
    <div class="logo-tx">Intelli<em>Doc</em></div></div>
    <div class="hsp"></div><span class="ver">v15</span><div class="sdot"></div>
  </div>
  <div id="idoc-body">
    <div id="idoc-sb">
      <div class="sb-sec">Sessions</div>
      <div class="sess-scroll">{sess_html}</div>
      <div class="sb-sec">Documents</div>
      <div class="file-scroll">{fhtml}</div>
      <div class="sb-foot"><div class="uav">U</div><div class="ulbl">{_e(mode)}</div></div>
    </div>
    <div id="idoc-main">
      <div id="idoc-topbar">
        <span class="sname">{_e(active_id)}</span>
        <span class="ssep">/</span><span class="sturns">{turns}t</span>
        <div class="hsp"></div>{fpills}
      </div>
      <div id="idoc-chat">{msgs}</div>
    </div>
  </div>
</div>
{_RESIZE_JS}"""

# ── Gradio app ────────────────────────────────────────────────────────────────
with gr.Blocks(theme=gr.themes.Base(), css=_CSS, title="IntelliDoc AI") as demo:

    app_state = gr.State(value=_new_app())
    chat_out  = gr.HTML(elem_id="idoc-chat-wrap", value="")

    # session-switch hidden textbox (still needed for JS → Python bridge)
    sw_inp = gr.Textbox(value="", visible=False, elem_id="sw-inp", interactive=True)

    with gr.Column(elem_id="idoc-bottom"):
        status_md = gr.Markdown("", elem_id="idoc-status")

        with gr.Row(elem_id="idoc-chips", equal_height=True):
            mode_radio = gr.Radio(
                choices=list(_MODE_MAP.keys()), value="Auto",
                label="", interactive=True)

        # ── FIX 1: real gr.Dropdown — no JS bridge, no hidden textbox ───────
        with gr.Row(elem_id="idoc-actions", equal_height=True):
            preload_dd = gr.Dropdown(
                choices=list(PRELOAD_CATALOG.keys()),
                value=None,
                label="",
                show_label=False,
                container=False,
                interactive=True,
                scale=3,
                elem_id="preload-dd",
            )
            preload_btn  = gr.Button("↓ Load",    scale=1, min_width=64)
            upload_btn   = gr.UploadButton(
                "↑ Upload", scale=1, min_width=64,
                file_types=[".pdf",".csv",".xlsx",".xls",".txt",".docx",".png",".jpg"],
                file_count="multiple", elem_classes=["upload-btn"])
            new_sess_btn = gr.Button("+ Session", scale=1, min_width=74)

        with gr.Row(elem_id="idoc-inp", equal_height=False):
            txt_in = gr.Textbox(
                placeholder="Ask anything about your document…",
                lines=1, max_lines=2,
                show_label=False, container=False, scale=9)
            with gr.Column(scale=0, min_width=74, elem_id="send-btn"):
                send_btn = gr.Button("Send →", variant="primary")

    # ── callbacks ─────────────────────────────────────────────────────────────

    demo.load(fn=lambda a: _render(a), inputs=[app_state], outputs=[chat_out])

    def on_switch(sid, app):
        sid = (sid or "").strip()
        if sid and sid in app["sessions"]:
            app = _set_active(app, sid)
        sess = _active(app)
        return app, _render(app), "", sess.get("mode", "Auto")

    sw_inp.change(fn=on_switch, inputs=[sw_inp, app_state],
                  outputs=[app_state, chat_out, sw_inp, mode_radio])

    def on_mode(mode, app):
        sess = dict(_active(app)); sess["mode"] = mode
        app  = _upsert(app, sess)
        return app, _render(app), f"mode → {mode}"

    mode_radio.change(fn=on_mode, inputs=[mode_radio, app_state],
                      outputs=[app_state, chat_out, status_md])

    def on_new(app):
        existing = set(app["sessions"].keys()); i = len(existing)+1
        while f"Chat {i}" in existing: i += 1
        ns  = _new_session(f"Chat {i}")
        app = _upsert(app, ns); app = _set_active(app, ns["id"])
        return app, _render(app), "", f"new: {ns['id']}"

    new_sess_btn.click(fn=on_new, inputs=[app_state],
                       outputs=[app_state, chat_out, txt_in, status_md])

    # ── FIX 1 continued: preload_dd value goes directly into on_preload ──────
    def on_preload(fname, app):
        fname = _safe(fname).strip()
        if not fname:
            return app, _render(app), "⚠ select a file from the dropdown first"
        fpath = PRELOAD_CATALOG.get(fname, "")
        if not fpath or not pathlib.Path(fpath).exists():
            return app, _render(app), f"✗ not found: {fname}"
        sess = dict(_active(app)); sess["files"] = dict(sess.get("files", {}))
        try:
            r = _ingest(fpath)
            source_type = _safe(r.get("source_type"), "TEXT_RAG") or "TEXT_RAG"
            p = _build(
                rag_index   = r.get("rag_index"),
                doc_index   = r.get("doc_index"),
                table_names = r.get("sql_tables"),
                source_type = source_type,
            )
            # Mirror inline tester: register_files so SQL/RAG nodes find the data
            import pandas as _pd
            df = None
            if fpath.endswith(".csv"):
                try:
                    df = _pd.read_csv(fpath, encoding="latin-1", on_bad_lines="skip")
                except Exception:
                    pass
            _register(sess["thread"], {
                fname: {"path": fpath, "text": "", "df": df}
            })
            sess["files"][fname] = {"path": fpath, "source_type": source_type}
            sess["pipeline"] = p
            app = _upsert(app, sess)
            return app, _render(app), f"✓ {fname}"
        except Exception as e:
            import traceback; traceback.print_exc()
            return app, _render(app), f"✗ {e}"

    # Load button reads preload_dd directly — no JS bridge needed
    preload_btn.click(fn=on_preload, inputs=[preload_dd, app_state],
                      outputs=[app_state, chat_out, status_md])

    def on_upload(files, app):
        if not files: return app, _render(app), "⚠ no files"
        sess = dict(_active(app)); sess["files"] = dict(sess.get("files", {}))
        loaded = []; last_p = sess.get("pipeline")
        for f in (files if isinstance(files, list) else [files]):
            fpath = f.name if hasattr(f, "name") else str(f)
            fname = pathlib.Path(fpath).name
            try:
                r = _ingest(fpath)
                source_type = _safe(r.get("source_type"), "TEXT_RAG") or "TEXT_RAG"
                p = _build(
                    rag_index   = r.get("rag_index"),
                    doc_index   = r.get("doc_index"),
                    table_names = r.get("sql_tables"),
                    source_type = source_type,
                )
                # FIX 2 consequence: if _ingest raises ValueError (empty chunks),
                # the except block below catches it and shows a friendly message.

                # Mirror inline tester: register_files so SQL/RAG nodes find data
                import pandas as _pd
                df = None
                if fpath.endswith(".csv"):
                    try:
                        df = _pd.read_csv(fpath, encoding="latin-1", on_bad_lines="skip")
                    except Exception:
                        pass
                _register(sess["thread"], {
                    fname: {"path": fpath, "text": "", "df": df}
                })
                sess["files"][fname] = {"path": fpath, "source_type": source_type}
                last_p = p; loaded.append(fname)
            except Exception as e:
                import traceback; traceback.print_exc()
                return app, _render(app), f"✗ {fname}: {e}"
        sess["pipeline"] = last_p; app = _upsert(app, sess)
        return app, _render(app), f"✓ {', '.join(loaded)}"

    upload_btn.upload(fn=on_upload, inputs=[upload_btn, app_state],
                      outputs=[app_state, chat_out, status_md])

    def on_send(question, app):
        question = _safe(question).strip()
        if not question:
            return app, _render(app), "", "⚠ type something"
        sess = _active(app)
        if not sess.get("pipeline"):
            return app, _render(app), "", "⚠ load a file first"
        sess    = dict(sess)
        history = list(sess.get("history", []))
        files   = sess.get("files", {})
        history.append({"role":"me", "text":question})
        sess["history"] = history
        t0 = time.perf_counter()
        try:
            force_route = _MODE_MAP.get(sess.get("mode","Auto"))  # None = Auto
            res = sess["pipeline"].invoke(
                {"user_query"  : question,
                 "_thread_id"  : sess["thread"],
                 "file_names"  : list(files.keys()),
                 "force_route" : force_route,
                 "chart_html"  : ""},
                _cfg(sess))
            ans        = _safe(res.get("final_response"))
            route      = _safe(res.get("route"))
            chart_html = _safe(res.get("chart_html"))
        except Exception as e:
            import traceback; traceback.print_exc()
            ans = "⚠ " + str(e); route = "error"; chart_html = ""
        ms = (time.perf_counter() - t0) * 1000
        # FIX 3: don't store is_html — _render() checks chart_html directly
        history.append({"role":"ai", "text":ans, "chart_html":chart_html,
                        "ms":ms, "route":route})
        sess["history"] = history; app = _upsert(app, sess)
        return app, _render(app), "", f"→ {route}  ·  {ms:.0f}ms"

    send_btn.click(fn=on_send, inputs=[txt_in, app_state],
                   outputs=[app_state, chat_out, txt_in, status_md])
    txt_in.submit(fn=on_send, inputs=[txt_in, app_state],
                  outputs=[app_state, chat_out, txt_in, status_md])

demo.queue().launch(inline=True)

